# The stop judge

At every decision point KORA holds a **state**: a question (or sub-question), the facts retrieved so far, and a
**candidate answer**. The **stop judge** reads the state and says *stop* (the candidate answers it) or *keep
searching*. The **oracle** makes the same call with the gold answer: yes when the candidate covers at least half the
gold and at least half of it is gold. We train a judge to agree with the oracle.

| step | section |
|---|---|
| exploratory analysis | 1 |
| splits | 2 |
| audit of the data | 3 |
| cleaning, pre-processing, the page (4.4) | 4 |
| feature engineering | 5 |
| model selection and evaluation | 6 |

Data: `DATA_DIR/raw/` (`DATA_DIR` is `ESTIMATOR_DATA_DIR`, default `estimator/data`). Outputs, under `DATA_DIR`:
`cleaned/`, `rendered/`, `validation.parquet`, `holdout.parquet`, and the split lists in `splits/`.

In [ ]:
import json, glob, os, re, ast, random, collections, textwrap
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chisquare

DATA_DIR = os.environ.get("ESTIMATOR_DATA_DIR", "estimator/data")
V = DATA_DIR; RAW = f"{V}/raw"; OUT = f"{V}/splits"
SETS = ["webqsp", "cwq", "grailqa", "metaqa", "biocq"]          # KORA has been run on these
ALL = SETS + ["kqapro"]
pd.set_option("display.max_colwidth", 100); pd.set_option("display.width", 200)

def jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def show(text, width=120):
    print("\n".join(textwrap.fill(l, width, subsequent_indent="      ") if len(l) > width else l for l in str(text).splitlines()))

norm = lambda t: re.sub(r"\s+", " ", str(t).strip().lower())

# questions and their gold
Q = {s: json.load(open(f"{RAW}/questions/{s}/train.json"))["questions"] for s in ALL}
QV = {"kqapro": json.load(open(f"{RAW}/questions/kqapro/val.json"))["questions"]}
QUESTION = {str(x["id"]): x for s in ALL for x in Q[s] + QV.get(s, [])}
BENCH = {str(x["id"]): s for s in ALL for x in Q[s] + QV.get(s, [])}
GOLD_IDS = {i: {str(a) for a in (x.get("answer_ids") or [])} for i, x in QUESTION.items()}
GOLD_NAMES = {i: {norm(a) for a in (x.get("answers") or []) if str(a).strip()} for i, x in QUESTION.items()}

## 1. Exploratory analysis

Two training sources, both on the same pool of training questions:

| | dataset | what it is |
|---|---|---|
| **A** | generated | built from the graph and the gold, no system run: the gold path(s) and near-miss paths (a sibling relation, a hop too many or too few, a dropped constraint, ...) |
| **B** | KORA harvest | KORA (Qwen3-30B) run with the **oracle** stopping: every state it judged |

Not used now: older harvested and generated datasets (in an old text format that cannot be re-rendered), and the
DAgger on-policy states (later). The
judge-server logs of the DAgger runs are loaded only as the reference for **what live text looks like**.

In [ ]:
A = []
for s in ALL:
    for a, b in zip(open(f"{RAW}/generated/{s}.rows.jsonl"), open(f"{RAW}/generated/{s}.meta.jsonl")):
        x, m = json.loads(a), json.loads(b)
        A.append({**m, "qid": str(m["qid"]), "text": x["state"], "label": int(x["expected"]["sufficient"])})
A = pd.DataFrame(A)
B = pd.concat([pd.DataFrame(jsonl(f"{RAW}/oracle_runs/{s}.jsonl")).assign(bench=s) for s in SETS], ignore_index=True)
B["qid"] = B.qid.astype(str); B["label"] = B.label.astype(int)
B["is_subquestion"] = [norm(g) != norm(QUESTION.get(q, {}).get("question", "")) for g, q in zip(B.goal, B.qid)]
LIVE = pd.concat([pd.DataFrame(jsonl(f)) for f in sorted(glob.glob(f"{RAW}/onpolicy/*.decisions.jsonl"))], ignore_index=True)
pd.DataFrame({"rows (decision states)": [len(A), len(B)], "questions": [A.qid.nunique(), B.qid.nunique()],
              "sets": [A.bench.nunique(), B.bench.nunique()], "yes share": [A.label.mean().round(3), B.label.mean().round(3)]},
             index=["A generated", "B KORA harvest"])

### 1.1 Dataset A: one row, and the kinds of rows
`spine` is the gold path; every other kind is a variation of it or another path the generator walked.

In [ ]:
r = A[(A.bench == "cwq") & (A.source == "sibling")].iloc[0]
print("kind:", r.source, "| label:", r.label, "| recall", r.recall, "precision", r.precision); show(r.text)

In [ ]:
A.groupby("source").agg(rows=("label", "size"), yes=("label", "mean")).sort_values("rows", ascending=False).round(3)

### 1.2 Dataset B: one state, and the sets
Each state is what KORA's judge was shown, in KORA's own words; the executed query and the candidate ids are kept
beside it. A **sub-question state** is one whose goal is a part of the question, not the question.

In [ ]:
r = B.iloc[3]
print("goal:", r.goal, "| site:", r.site, "| query:", r.query); print("ids:", r.ids, "| label:", r.label); show(r.state)

In [ ]:
t = B.groupby("bench").agg(states=("qid", "size"), questions=("qid", "nunique"), yes=("label", "mean"),
                           subquestion=("is_subquestion", "mean"), root=("site", lambda s: (s == "root").mean()))
t.round(3)

### 1.3 Does A's text look like live text?
The judge is trained on text and served text, so the two must look alike. Live text = what the judge server wrote
during KORA runs.

In [ ]:
HEAD = "EVIDENCE (facts retrieved so far, as subject | relation | object)"
def shape(t):
    ev = t.split(HEAD, 1)[1].split("CANDIDATE ANSWER", 1)[0] if HEAD in t else ""
    ev = [l.strip() for l in ev.splitlines() if l.strip() and l.strip() != "(nothing retrieved)"]
    return {"characters": len(t), "evidence lines": len(ev), "a list as a subject": any(", " in l.split(" | ")[0] for l in ev),
            "a duplicated line": len(ev) != len(set(ev))}
pd.DataFrame({"A generated": pd.DataFrame([shape(t) for t in A.text]).mean(),
              "live": pd.DataFrame([shape(t) for t in LIVE.text]).mean()}).round(3)

**What the analysis shows**
- A is twice B's size (118k vs 55k states); both are about 20% yes.
- **A's text is not written like live text** (longer, lists as subjects, duplicated lines): it was written by the
  generator's own code, not the judge server's. Fixed in section 4 by rendering A with the server's function.
- **B's sub-question states (26%) are almost all labelled no**, because the oracle scored them against the whole
  question's gold (section 3 checks this).

## 2. Splits

Splits are made at the **question** level, before any cleaning or feature fitting, so nothing fitted to the data ever
sees a validation, holdout or test question. All states of a question are in one split.

| split | questions | how it was drawn | used for |
|---|---|---|---|
| **train** | ~5,800 | the base draw: 1,000 random questions per set from each set's train split, minus every question in validation, the retired holdout or test | the only questions training rows come from |
| **validation** | 666 | KORA's development set from the train splits, *type-balanced* (at least 6 questions of every type), two thirds of it | every choice: model, recipe, checkpoint, threshold |
| **holdout** | 1,200 (200 per set) | drawn **in proportion to each set's train split by question type**, from questions no source has used; KQA Pro: its own 200 validation questions (already proportional) | an unbiased check of the chosen model, once per experiment; never chosen on |
| **retired** | 334 | the old holdout (the other third of the development set): type-balanced, so not representative | nothing (never trained on, never read) |
| **test** | the benchmarks' test sets | published | the paper's numbers, once at the very end, scores only |

Validation stays type-balanced on purpose (every rare type present when choosing); the holdout estimates performance
on the real mix. KQA Pro has no validation questions yet (it joins with its own KORA runs).

In [ ]:
def qid_set(name):
    d = json.load(open(f"{RAW}/qids/{name}.json"))
    items = d["questions"] if isinstance(d, dict) and "questions" in d else d
    return {str(x["id"]) if isinstance(x, dict) else str(x) for x in items}

TEST = json.load(open(f"{RAW}/test_questions.json"))            # ids and texts only, never answers
test_ids = {x["id"] for v in TEST.values() for x in v}
test_txt = {norm(x["question"]) for v in TEST.values() for x in v if x["question"]}

def gold_query(x):
    g = x.get("gold")
    if isinstance(g, str):
        try:
            g = ast.literal_eval(g)
        except Exception:
            return ""
    return g.get("query", "") if isinstance(g, dict) else ""

def categories(s, x):
    """The question types each set carries (2.2); WebQSP and CWQ's derived from the gold SPARQL."""
    c = {}
    if s in ("webqsp", "cwq"):
        q = gold_query(x); body = q.split("WHERE", 1)[-1]
        ents = set(re.findall(r"ns:(m\.[0-9a-z_]+)", re.sub(r"FILTER\s*\(\s*\?\w+\s*!=\s*ns:m\.[0-9a-z_]+\s*\)", "", body)))
        c["entities in query"] = "3+" if len(ents) >= 3 else str(len(ents))
        c["superlative"] = "ORDER BY" in q.upper()
        c["comparative"] = bool(re.search(r"FILTER\s*\([^)]*[<>]", q))
        c["triple patterns"] = str(min(len(re.findall(r"ns:[a-z_]+\.[a-z_0-9]+\.[a-z_0-9.]+", body)), 4))
    elif s == "grailqa":
        c["hops"] = str(x["hops"]); c["function"] = "comparative" if x["function"] in ("<", "<=", ">", ">=") else x["function"]
    elif s == "metaqa":
        c["hops"] = str(x["hops"])
    elif s == "biocq":
        c["hops"] = str(min(int(x["hops"]), 3)); c["question_type"] = x["question_type"]; c["kg_source"] = x["kg_source"]
    elif s == "kqapro":
        c["question_type"] = x["question_type"]
    return c

def stratum(s, x):
    """The holdout's strata: those categories, combined coarsely so no cell is empty."""
    c = categories(s, x)
    if s in ("webqsp", "cwq"):
        return ("superlative" if c["superlative"] else "comparative" if c["comparative"] else
                "multi-entity" if c["entities in query"] in ("2", "3+") else
                "chain" if c["triple patterns"] in ("2", "3", "4") else "1 triple")
    if s == "grailqa":
        return f'{min(int(c["hops"]), 3)} hops | {"function" if c["function"] != "none" else "no function"}'
    if s == "metaqa":
        return f'{c["hops"]} hops'
    if s == "biocq":
        return f'{c["kg_source"]} | {c["question_type"]} | {c["hops"]} hops'
    return c.get("question_type", "")

validation, retired = qid_set("V_tune"), qid_set("V_hold")
base = set().union(*(qid_set(f"T_{s}") for s in ALL))
used = base | validation | retired | qid_set("T_r1") | qid_set("T_r2") | qid_set("T_r3")
in_test = {i for i, x in QUESTION.items() if norm(x["question"]) in test_txt}

HOLD_SIZE, SEED = 200, 20261016
def proportional_draw(s):
    """200 questions of set s in proportion to its train split's strata (largest remainder), from unused questions."""
    rng = random.Random(f"{SEED}-{s}")
    pop = collections.Counter(stratum(s, x) for x in Q[s])
    free = collections.defaultdict(list)
    for x in Q[s]:
        i = str(x["id"])
        if i not in used and i not in in_test and (GOLD_IDS[i] or GOLD_NAMES[i]):
            free[stratum(s, x)].append(i)
    quota = {k: HOLD_SIZE * n / len(Q[s]) for k, n in pop.items()}
    alloc = {k: int(v) for k, v in quota.items()}
    for k in sorted(quota, key=lambda k: -(quota[k] - alloc[k]))[:HOLD_SIZE - sum(alloc.values())]:
        alloc[k] += 1
    out, short = [], 0
    for k in sorted(alloc):
        pool = sorted(free[k]); take = min(alloc[k], len(pool)); short += alloc[k] - take
        out += rng.sample(pool, take)
    if short:                                   # a stratum short of unused questions: fill from the rest at random
        rest = sorted({i for v in free.values() for i in v} - set(out))
        out += rng.sample(rest, min(short, len(rest)))
    return set(out), short

holdout, shortfall = set(qid_set("kqapro_val")), {}
for s in SETS:
    h, shortfall[s] = proportional_draw(s); holdout |= h
train = base - validation - retired - holdout - in_test
SPLITS = {"train": train, "validation": validation, "holdout": holdout, "retired": retired}
tab = pd.DataFrame({k: collections.Counter(BENCH[i] for i in v) for k, v in SPLITS.items()}).reindex(ALL).fillna(0).astype(int)
tab.loc["total"] = tab.sum()
print(f"base draw {len(base)}: minus {len(base & validation)} validation, {len(base & retired)} retired, {len(base & in_test)} with a test question's text")
print("holdout strata short of unused questions (filled at random):", {k: v for k, v in shortfall.items() if v} or "none")
tab

In [ ]:
# Checks: the splits are disjoint by id and by text, and none holds a test question's text.
names = list(SPLITS)
text_of = lambda ids: {norm(QUESTION[i]["question"]) for i in ids}
checks = [(f"{a} / {b}: shared ids", len(SPLITS[a] & SPLITS[b])) for i, a in enumerate(names) for b in names[i + 1:]]
checks += [(f"{a} / {b}: shared texts", len(text_of(SPLITS[a]) & text_of(SPLITS[b]))) for i, a in enumerate(names) for b in names[i + 1:]]
checks += [(f"{a}: test texts", len(text_of(v) & test_txt)) for a, v in SPLITS.items()]
checks += [("train: questions without a gold answer", sum(not (GOLD_IDS[i] or GOLD_NAMES[i]) for i in train))]
pd.DataFrame(checks, columns=["check", "count (must be 0)"])

In [ ]:
os.makedirs(f"{OUT}", exist_ok=True)
for k in SPLITS:                                    # only the files written here (round and screen lists stay)
    if os.path.exists(f"{OUT}/{k}.json"):
        os.remove(f"{OUT}/{k}.json")
for k, v in SPLITS.items():
    json.dump({"split": k, "questions": sorted(v), "made": "2026-10-16", "source": "stop_judge.ipynb section 2"},
              open(f"{OUT}/{k}.json", "w"))
print(sorted(os.listdir(f"{OUT}")))

### 2.2 Are validation and holdout representative of the train distribution?
Validation is type-balanced by design; the holdout is drawn to be proportional, and this checks both.
Each slice against its set's train split, category by category (chi-square goodness of fit, small cells pooled;
p < .05 = a kind of question is over- or under-represented). Categories: hops, function and question type where the
records carry them; for WebQSP and CWQ, derived from the gold SPARQL (superlative = `ORDER BY`, comparative =
`FILTER <,>`, entities in the query, triple patterns).

In [ ]:
def fit(pop, obs):
    keys = sorted(pop, key=lambda k: -pop[k]); n, N = sum(obs.values()), sum(pop.values())
    exp = {k: pop[k] / N * n for k in keys}
    big = [k for k in keys if exp[k] >= 5]; small = [k for k in keys if exp[k] < 5]
    o, e = [obs.get(k, 0) for k in big], [exp[k] for k in big]
    if small:
        o.append(sum(obs.get(k, 0) for k in small)); e.append(sum(exp[k] for k in small))
    return chisquare(o, e).pvalue if len(o) > 1 else float("nan")

rows = []
for s in ALL:
    pop_c = [categories(s, x) for x in Q[s]]
    slices = {k: [x for x in Q[s] + QV.get(s, []) if str(x["id"]) in SPLITS[k]] for k in ("validation", "holdout")}
    for var in pop_c[0]:
        pop = collections.Counter(c[var] for c in pop_c)
        for name, xs in slices.items():
            if not xs:
                continue
            obs = collections.Counter(categories(s, x)[var] for x in xs)
            gap = max(abs(obs.get(k, 0) / len(xs) - pop[k] / len(pop_c)) for k in pop)
            rows.append({"set": s, "category": var, "slice": name, "n": len(xs), "p": round(fit(pop, obs), 3), "largest gap (points)": round(100 * gap, 1)})
rep = pd.DataFrame(rows)
rep.pivot_table(index=["set", "category"], columns="slice", values=["p", "largest gap (points)"], sort=False)

**Reading:** the **holdout is representative**: in every set and category p > .2 and the largest gap is under 6
points (CWQ superlatives 7% as in train). **Validation is not, by design**: CWQ and GrailQA over-represent the
harder types (CWQ superlatives about 22% vs 7%; GrailQA 3–4 hops about 23% vs 5%), WebQSP close, MetaQA and BioCQ
representative. Choosing on a type-balanced validation and estimating on a proportional holdout; per-type results
reported throughout.

## 3. Audit of A and B

Read-only: is every row complete, and is every label right? Format does not matter here (section 4). For each row:
the question, the path it took, the candidate it got, and its label. Both datasets' labels come from the **ground
truth**: A's from the gold answers when it was generated, B's from the oracle, which is the gold answers too. So each
label is recomputed here from the candidate and the gold, by the oracle's rule, and compared with the stored one.

| check | why |
|---|---|
| 3.1 complete: question, path, candidate, label present; the question has gold; which split it belongs to | a row missing a part, or on a held-out question, cannot be trained on |
| 3.2 the stored label follows the rule from the stored recall and precision | the rule was applied as written |
| 3.3 the label recomputed from the candidate and the gold | the stored recall and precision were right |
| 3.4 sub-question states | labelled against the whole question's gold (B) |
| 3.5 duplicates and identical states with different labels | inflate a question, or undecidable |

In [ ]:
def b_evidence(state):
    if "EVIDENCE" not in state:
        return []
    return [l.strip() for l in state.split("EVIDENCE (the path that was traversed)")[-1].split("CANDIDATE ANSWER")[0].splitlines() if l.strip()]

def split_of(q):
    return next((k for k, v in SPLITS.items() if q in v), "outside the splits")

audit_A = pd.DataFrame({"qid": A.qid, "bench": A.bench, "question": A.question, "path": A.evidence,
                        "candidate": A.candidate, "n_candidate": A.n_candidate, "label": A.label})
audit_B = pd.DataFrame({"qid": B.qid, "bench": B.bench, "question": B.goal, "path": B.state.map(b_evidence),
                        "candidate": B.ids, "n_candidate": B.n_got, "label": B.label})
def completeness(df):
    return {"rows": len(df),
            "no question": int((df.question.fillna("").str.strip() == "").sum()),
            "no path": int((df.path.map(len) == 0).sum()),
            "no candidate": int((df.candidate.map(lambda c: len(c or [])) == 0).sum()),
            "label not 0/1": int((~df.label.isin([0, 1])).sum()),
            "question has no gold": int(sum(not (GOLD_IDS.get(q) or GOLD_NAMES.get(q)) for q in df.qid)),
            **{f"on {k}": int(v) for k, v in df.qid.map(split_of).value_counts().items()}}
pd.DataFrame({"A": completeness(audit_A), "B": completeness(audit_B)}).fillna(0).astype(int)

In [ ]:
# 3.2 The stored label equals the rule (recall >= .5 and precision >= .5) on the stored recall and precision.
rule = lambda d: ((d.recall >= .5) & (d.precision >= .5)).astype(int)
pd.DataFrame({"A": {"rows": len(A), "label differs from the rule": int((A.label != rule(A)).sum())},
              "B": {"rows": len(B), "label differs from the rule": int((B.label != rule(B)).sum())}})

In [ ]:
# 3.3 The label recomputed from the candidate and the gold (ids where the gold has ids; names otherwise).
def oracle(cand_ids, cand_names, q):
    gids, gnm = GOLD_IDS.get(q, set()), GOLD_NAMES.get(q, set())
    if not cand_ids and not cand_names:
        return 0, 0.0, 0.0
    n = max(len(cand_ids), len(cand_names))
    hit_i = {c for c in cand_ids if c in gids}
    hit_n = {norm(c) for c in list(cand_names) + list(cand_ids) if norm(c) in gnm}
    matched = sum(1 for k in range(n) if (k < len(cand_ids) and (cand_ids[k] in gids or norm(cand_ids[k]) in gnm))
                  or (k < len(cand_names) and norm(cand_names[k]) in gnm))
    total = max(len(gids), len(gnm)) or 1
    rec, prec = max(len(hit_i), len(hit_n)) / total, matched / n
    return int(rec >= .5 and prec >= .5), rec, prec

# A carries candidate names only (its ids were not kept): precision over its true candidate count.
A["label_recomputed"] = [oracle([], list(c), q)[0] if len(c) == n else
                         int(oracle([], list(c), q)[1] >= .5 and sum(norm(x) in GOLD_NAMES.get(q, set()) for x in c) / max(n, 1) >= .5)
                         for c, n, q in zip(A.candidate, A.n_candidate, A.qid)]
B["label_recomputed"] = [oracle([str(i) for i in (ids or [])], list(nm or []) if len(nm or []) == len(ids or []) else [], q)[0]
                         for ids, nm, q in zip(B.ids, B.names, B.qid)]
agree = pd.concat([A.assign(dataset="A"), B.assign(dataset="B")]).groupby(["dataset", "bench"]).apply(
    lambda d: pd.Series({"rows": len(d), "agree": (d.label == d.label_recomputed).mean(),
                         "stored yes, recomputed no": int(((d.label == 1) & (d.label_recomputed == 0)).sum()),
                         "stored no, recomputed yes": int(((d.label == 0) & (d.label_recomputed == 1)).sum())}))
agree.round(4)

In [ ]:
# A few disagreements, to read.
for name, d in (("A", A), ("B", B)):
    bad = d[d.label != d.label_recomputed]
    print(f"===== {name}: {len(bad)} disagreements")
    for _, r in bad.head(3).iterrows():
        q = r.qid
        print("question:", QUESTION[q]["question"][:110]); print("   gold:", sorted(GOLD_NAMES[q])[:6], sorted(GOLD_IDS[q])[:4])
        print("   candidate:", (list(r.candidate) if name == "A" else list(r.ids))[:6], "| stored", r.label, "recomputed", r.label_recomputed)

In [ ]:
# 3.4 Sub-question states in B: the oracle labelled them against the question's gold, not the sub-question's.
B.groupby(["is_subquestion", "site"]).agg(states=("qid", "size"), yes=("label", "mean")).round(3)

In [ ]:
# 3.5 Duplicates (same question, same state) and identical states with different labels.
def dup(df, key):
    k = df.qid + "\u0001" + df[key].astype(str)
    conflict = df.groupby(df[key].astype(str)).label.nunique()
    return {"rows": len(df), "duplicate rows": int(k.duplicated().sum()), "states with both labels": int((conflict > 1).sum())}
pd.DataFrame({"A": dup(A, "text"), "B": dup(B.assign(key=B.goal.astype(str) + B.state), "key")})

### 3.6–3.9 Against the graph and the runs
The checks above take each row's candidate as given. These test the candidate itself, and B's runs:

| check | how |
|---|---|
| 3.6 A's paths re-executed | a sample of A's paths (60 per set, no constraints, no joined hops) walked again on the live graph with queries written here, not the generator's code: same candidates? the label recomputed **by id** |
| 3.7 B's queries re-executed | a sample of B's plain chain queries (60 per set) run again: same candidate ids? |
| 3.8 B's runs complete | each base question's harness row from the oracle runs: finished, errors, its states logged |
| 3.9 near-duplicate questions | train against validation, holdout and test by text similarity, not only exact text |

KQA Pro's graph server is down, so 3.6 covers the other five sets.

In [ ]:
# Graph access, written here: one hop from a set of nodes, and the names of nodes.
import urllib.request, urllib.parse
from neo4j import GraphDatabase, Query
from concurrent.futures import ThreadPoolExecutor, as_completed
POOL, QUERY_TIMEOUT = 32, 30            # parallel queries; seconds per query (a slower one counts as timed out)
EP = {k: open(f"{DATA_DIR}/runtime/endpoints/{k}.txt").read().split()[0] for k in ("freebase", "biokg", "metaqa")}
FB, BIO = "http://rdf.freebase.com/ns/", "http://skill-lab-research.org"
NEO = GraphDatabase.driver(EP["metaqa"], auth=(os.environ.get("NEO4J_USER", "neo4j"), os.environ.get("NEO4J_PASSWORD", "korakora")))   # KORA's defaults

def sparql(url, q, timeout=QUERY_TIMEOUT):
    req = urllib.request.Request(url, data=urllib.parse.urlencode({"query": q}).encode(),
                                 headers={"Accept": "application/sparql-results+json"})
    return json.load(urllib.request.urlopen(req, timeout=timeout))["results"]["bindings"]

def cypher(q, **kw):
    with NEO.session(database="neo4j") as s:
        return [r.data() for r in s.run(Query(q, timeout=QUERY_TIMEOUT), **kw)]

bio_uri = lambda i: i if i.startswith("http") else f"{BIO}/{i.split(':', 1)[0].lower()}/node/{i}"
bio_id = lambda u: u.rsplit("/node/", 1)[-1] if "/node/" in u else u
def token_of(p):                     # a BioKG predicate URI as the relation token the data writes
    g, _, n = p.partition("/relation/")
    return f"{g.rsplit('/', 1)[-1]}.{re.sub(r'[^A-Za-z0-9]+', '_', n)}" if n else p

def outcome_of(e):
    msg = f"{type(e).__name__} {e}".lower()
    return "timed out" if ("timed out" in msg or "timeout" in msg) else f"query error {type(e).__name__}"

def run_parallel(fn, rows, label):
    """fn over rows, POOL at a time; progress every 50."""
    out = []
    with ThreadPoolExecutor(POOL) as ex:
        futs = [ex.submit(fn, r) for r in rows]
        for k, f in enumerate(as_completed(futs), 1):
            out.append(f.result())
            if k % 50 == 0 or k == len(futs):
                print(f"{label}: {k}/{len(futs)}", flush=True)
    return out

def follow(bench, ids, rel, direction, limit=200):
    """The nodes reached from `ids` through `rel` (out: ids -rel-> x; in: x -rel-> ids)."""
    ids = list(dict.fromkeys(ids))[:2000]
    if not ids:
        return []
    if bench in ("webqsp", "cwq", "grailqa"):
        vals = " ".join(f"ns:{i}" for i in ids if re.fullmatch(r"[mg]\.[0-9a-z_]+", i))
        if not vals:
            return []
        pat = f"?s ns:{rel} ?t" if direction == "out" else f"?t ns:{rel} ?s"
        rows = sparql(EP["freebase"], f"PREFIX ns: <{FB}> SELECT DISTINCT ?t WHERE {{ VALUES ?s {{ {vals} }} {pat} }} LIMIT {limit}")
        return [b["t"]["value"].replace(FB, "") for b in rows]
    if bench == "metaqa":
        r = re.sub(r"[^A-Za-z0-9_]", "", rel)
        pat = f"(n)-[:`{r}`]->(m:Entity)" if direction == "out" else f"(n)<-[:`{r}`]-(m:Entity)"
        return [x["id"] for x in cypher(f"MATCH (n:Entity) WHERE n.id IN $ids MATCH {pat} RETURN DISTINCT m.id AS id LIMIT {limit}", ids=ids)]
    if bench == "biocq":
        vals = " ".join(f"<{bio_uri(i)}>" for i in ids)
        pat = "?s ?p ?t . FILTER(isIRI(?t))" if direction == "out" else "?t ?p ?s"
        rows = sparql(EP["biokg"], f"SELECT DISTINCT ?p ?t WHERE {{ VALUES ?s {{ {vals} }} {pat} }} LIMIT 100000")
        return list(dict.fromkeys(bio_id(b["t"]["value"]) for b in rows if token_of(b["p"]["value"]) == rel))[:limit]
    raise ValueError(bench)

def names(bench, ids):
    ids = list(dict.fromkeys(ids))
    if bench == "metaqa":
        return {i: i for i in ids}
    out = {}
    if bench == "biocq":
        for k in range(0, len(ids), 200):
            vals = " ".join(f"<{bio_uri(i)}>" for i in ids[k:k + 200])
            for b in sparql(EP["biokg"], f"SELECT ?s ?n WHERE {{ VALUES ?s {{ {vals} }} ?s <http://www.w3.org/2000/01/rdf-schema#label> ?n }}"):
                out[bio_id(b["s"]["value"])] = b["n"]["value"]
        return out
    ents = [i for i in ids if re.fullmatch(r"[mg]\.[0-9a-z_]+", i)]
    for k in range(0, len(ents), 200):
        vals = " ".join(f"ns:{i}" for i in ents[k:k + 200])
        q = f"PREFIX ns: <{FB}> SELECT ?s ?n WHERE {{ VALUES ?s {{ {vals} }} ?s ns:type.object.name ?n . FILTER(lang(?n) = 'en') }}"
        for b in sparql(EP["freebase"], q):
            out[b["s"]["value"].replace(FB, "")] = b["n"]["value"]
    out.update({i: i for i in ids if i not in ents})                 # a literal names itself
    return out

print({k: v for k, v in EP.items()}, "| freebase answers:", bool(sparql(EP["freebase"], "SELECT * WHERE { ?s ?p ?o } LIMIT 1")),
      "| metaqa answers:", bool(cypher("MATCH (n:Entity) RETURN n.id LIMIT 1")))

In [ ]:
# 3.6 A's paths re-executed: start from the question's topic entities named on the path's first line, follow each
# relation in its direction, compare the reached set with the stored candidate, and recompute the label by id.
def topic(q):
    te = QUESTION[q].get("topic_entity") or {}
    return ast.literal_eval(te) if isinstance(te, str) else te

DIRECTION = {">": "out", "<": "in", "out": "out", "in": "in"}     # A writes hop directions as > and <

def start_ids(r):
    first = r.evidence[0].split(" | ")
    side = first[0] if DIRECTION[r.directions[0]] == "out" else first[-1]
    want = {norm(x) for x in side.split(", ")}
    return [i for i, n in topic(r.qid).items() if norm(n) in want]

ok_rel = lambda rs: bool(rs) and not any(" & " in x or " / " in x for x in rs)
eligible_A = A[(A.bench != "kqapro") & (A.constraints == 0) & A.relations.map(ok_rel) & (A.qid.isin(train))]
sample_A = pd.concat([g.sample(min(60, len(g)), random_state=0) for _, g in eligible_A.groupby("bench")])
def check_A(r):
    cur, cut = start_ids(r), False
    if not cur:
        return {"bench": r.bench, "kind": r.source, "outcome": "no named start"}
    try:
        for rel, d in zip(r.relations, r.directions):
            cur = follow(r.bench, cur, rel, DIRECTION[d]); cut |= len(cur) >= 200
        cur = list(dict.fromkeys(cur)); nm = names(r.bench, cur)
    except Exception as e:
        return {"bench": r.bench, "kind": r.source, "outcome": outcome_of(e)}
    got, want = {norm(n) for n in nm.values() if n}, {norm(c) for c in r.candidate}
    lab = oracle(cur, [nm.get(i, "") for i in cur], r.qid)[0]
    return {"bench": r.bench, "kind": r.source, "outcome": "hit the 200 limit" if cut else ("same" if got == want else "different"),
            "stored n": r.n_candidate, "re-run n": len(cur), "label stored": r.label, "label by id": lab, "qid": r.qid,
            "missing": sorted(want - got)[:3], "extra": sorted(got - want)[:3]}
res = run_parallel(check_A, [r for _, r in sample_A.iterrows()], "3.6")
R6 = pd.DataFrame(res)
print(f"eligible rows: {len(eligible_A)} of {len(A)}; sampled {len(R6)}")
cmp6 = R6[R6.outcome.isin(["same", "different"])]
pd.concat([R6.groupby("bench").outcome.value_counts().unstack(fill_value=0),
           cmp6.groupby("bench").apply(lambda d: pd.Series({"label by id = stored": (d["label stored"] == d["label by id"]).mean().round(3)}))], axis=1)

In [ ]:
# Reading 3.6: the rows that came out different, and the label disagreements among the same ones.
show_cols = ["bench", "kind", "outcome", "stored n", "re-run n", "label stored", "label by id", "missing", "extra"]
print(R6[R6.outcome == "different"][show_cols].head(8).to_string())
print(R6[(R6.outcome == "same") & (R6["label stored"] != R6["label by id"])][show_cols].head(5).to_string())

In [ ]:
# 3.7 B's queries re-executed: plain chain queries (one constant, each variable reached by one triple, no restriction
# words, no ORDER BY), run again triple by triple; the reached ids compared with the ids B logged.
TOK = re.compile(r'"[^"]*"|<[^>]*>|\S+')
def triples_of(q):
    i, j = q.find("{"), q.rfind("}")
    parts = [TOK.findall(p) for p in re.split(r"\s\.\s|\s\.$", " " + q[i + 1:j].strip() + " ")]
    return [tuple(p) for p in parts if len(p) == 3]

def plain_chain(q):
    if not isinstance(q, str) or not q or "ORDER" in q.upper() or "LIMIT" in q.upper():
        return False                                # root states carry no query
    ts = triples_of(q)
    if not ts or any(p.startswith(":") for _, p, _ in ts):
        return False
    variables = {t for s, _, o in ts for t in (s, o) if t.startswith("?")}
    constants = [t for s, _, o in ts for t in (s, o) if not t.startswith("?")]
    return len(ts) == len(variables) and len(constants) == 1

def run_chain(bench, q):
    sel = re.search(r"SELECT\s+(?:DISTINCT\s+)?(\?\w+)", q).group(1)
    const = lambda t: bio_id(t.strip('"<>')) if bench == "biocq" else t.strip('"')
    var = lambda t: t.startswith("?")
    bound, todo = {}, triples_of(q)
    for _ in range(len(todo) + 1):
        progressed = False
        for t in list(todo):
            s, p, o = t
            if not var(s) and var(o) and o not in bound:   bound[o] = follow(bench, [const(s)], p, "out", 2000)
            elif not var(o) and var(s) and s not in bound: bound[s] = follow(bench, [const(o)], p, "in", 2000)
            elif s in bound and var(o) and o not in bound: bound[o] = follow(bench, bound[s], p, "out", 2000)
            elif o in bound and var(s) and s not in bound: bound[s] = follow(bench, bound[o], p, "in", 2000)
            else:
                continue
            todo.remove(t); progressed = True
        if not todo or not progressed:
            break
    return bound.get(sel, [])

eligible_B = B[B["query"].map(plain_chain) & B.qid.isin(train)]
sample_B = pd.concat([g.sample(min(60, len(g)), random_state=0) for _, g in eligible_B.groupby("bench")])
def check_B(r):
    try:
        got = set(run_chain(r.bench, r["query"]))
    except Exception as e:
        return {"bench": r.bench, "outcome": outcome_of(e)}
    want = {str(i) for i in r.ids}
    return {"bench": r.bench, "outcome": "hit the limit" if len(got) >= 2000 else ("same" if got == want else "different"),
            "logged n": len(want), "re-run n": len(got), "query": r["query"][:110],
            "missing": sorted(want - got)[:3], "extra": sorted(got - want)[:3]}
res = run_parallel(check_B, [r for _, r in sample_B.iterrows()], "3.7")
R7 = pd.DataFrame(res)
print(f"eligible states: {len(eligible_B)} of {len(B)}; sampled {len(R7)}")
R7.groupby("bench").outcome.value_counts().unstack(fill_value=0)

In [ ]:
print(R7[R7.outcome == "different"][["bench", "logged n", "re-run n", "missing", "extra", "query"]].head(8).to_string())

In [ ]:
# 3.8 B's runs complete: the oracle runs' own harness rows (results/eval/train-kora_oracle-qwen3-30b-a3b-<set>).
H = {}
for s in SETS:
    for f in glob.glob(f"{DATA_DIR}/results/eval/train-kora_oracle-qwen3-30b-a3b-{s}/qwen3-30b-a3b/{s}-train/kora_oracle/s*.jsonl"):
        if f.endswith("failed.jsonl"):
            continue
        for line in open(f):
            if line.strip():
                r = json.loads(line); fm = r.get("fw_metrics") or {}
                H[str(r["qid"])] = {"bench": s, "error": bool(r.get("error")), "status": (fm.get("kora") or {}).get("status"),
                                    "calls": {k: v.get("calls", 0) for k, v in (fm.get("breakdown_by_call_site") or {}).items()}}
base5 = {i for i in base if BENCH[i] in SETS}
states = B.groupby("qid").size()
comp = pd.DataFrame({s: {"base questions": sum(BENCH[i] == s for i in base5),
                         "with a harness row": sum(BENCH[i] == s and i in H for i in base5),
                         "errored": sum(BENCH[i] == s and H.get(i, {}).get("error", False) for i in base5),
                         "with states in B": sum(BENCH[i] == s and i in states.index for i in base5),
                         "states per question (median)": float(states[[i for i in base5 if BENCH[i] == s and i in states.index]].median())}
                     for s in SETS}).T
print("run statuses:", collections.Counter(h["status"] for h in H.values()).most_common())
print("B states whose question has no harness row:", int(sum(q not in H for q in B.qid)))
comp

In [ ]:
# Were all the judge's decisions logged? The run counts its own judge calls (call sites named *judge* / *suffic*);
# compare with the states B holds for the question (the oracle decided, but the judge was still asked every time).
sites = collections.Counter(k for h in H.values() for k in h["calls"])
judge_sites = [k for k in sites if "judge" in k or "suffic" in k]
print("call sites seen:", sorted(sites)); print("counted as judge calls:", judge_sites)
cmp8 = pd.DataFrame([{"bench": h["bench"], "judge calls": sum(h["calls"].get(k, 0) for k in judge_sites), "states logged": int(states.get(q, 0))}
                     for q, h in H.items() if not h["error"]])
cmp8["logged / calls"] = cmp8["states logged"] / cmp8["judge calls"].replace(0, np.nan)
cmp8.groupby("bench").agg(questions=("bench", "size"), calls=("judge calls", "sum"), logged=("states logged", "sum"),
                          ratio_median=("logged / calls", "median"), no_state_logged=("states logged", lambda s: int((s == 0).sum())))

In [ ]:
# 3.9 Near-duplicate questions: each validation, holdout and test question against its most similar train question
# (TF-IDF over words and word pairs, cosine). Template sets (MetaQA, BioCQ) are similar by construction; what would
# matter is the same question reworded, which the examples show.
from sklearn.feature_extraction.text import TfidfVectorizer
tr_ids = sorted(train); tr_txt = [QUESTION[i]["question"] for i in tr_ids]
held = {"validation": [QUESTION[i]["question"] for i in sorted(validation)], "holdout": [QUESTION[i]["question"] for i in sorted(holdout)],
        "test": [x["question"] for v in TEST.values() for x in v if x["question"]]}
vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(tr_txt + sum(held.values(), []))
Xtr = vec.transform(tr_txt)
near, ex = {}, []
for name, txt in held.items():
    S = vec.transform(txt) @ Xtr.T
    best = S.max(axis=1).toarray().ravel(); arg = np.asarray(S.argmax(axis=1)).ravel()
    near[name] = {"questions": len(txt), "cosine ≥ 0.95": int((best >= .95).sum()), "cosine ≥ 0.9": int((best >= .9).sum()),
                  "cosine ≥ 0.8": int((best >= .8).sum())}
    for k in np.argsort(-best)[:4]:
        ex.append({"held-out set": name, "cosine": round(float(best[k]), 3), "held-out question": txt[k][:80], "train question": tr_txt[arg[k]][:80]})
display(pd.DataFrame(near).T)
pd.DataFrame(ex)

**What the audit shows**
- **Complete:** every row has a question and a path, every label is 0/1, every question has gold. A has **2,817 rows
  with no named candidate** (paths ending on unnamed nodes; their text reads "(N items)" then "(empty)"): KORA never
  sends such a state, so they are dropped in section 4.
- **On held-out questions:** none in A (the generator excluded validation); in B **1,098 rows on validation, 644 on
  the retired old holdout and 14 on the one question that shares a test question's text**: dropped in section 4, leaving 53,076.
- **The rule was applied as written:** in both, every stored label equals "recall ≥ .5 and precision ≥ .5" on the
  stored recall and precision.
- **The labels match the ground truth:** recomputed from the candidate and the gold, they agree on **99.8%+ of rows
  in every set** (A 98 rows differ, B 25). A's differences are matches the generator made **by id** that a
  name-only recomputation misses (A kept names, not ids: "Nicomachean ethics" vs the gold's name variants), so A's
  stored label is the right one. B's are **same-name, different-entity** cases (a candidate named like the gold
  answer but with another Freebase id): the oracle scored by id, so no. 25 rows; kept as the oracle labelled them.
- **Sub-question states (B):** 14,185 states, 2.1% yes against 18.7% for full-question leaves, because they were
  scored against the whole question's gold. Dropped in section 4, in round 0 and in every later round.
- **Duplicates:** 185 in A, 1,164 in B (the same state judged twice in a run); removed in section 4. No identical
  state carries both labels.

**Against the graph and the runs (3.6–3.9)**
- **3.6 A's paths reproduce on the graph.** Of 300 sampled paths (60 per set; KQA Pro's graph is down), 273 could be
  compared exactly: **270 return the stored candidates (98.9%)**; 25 more hit the 200-node follow limit (the generator
  kept the first 200, so the sets are not comparable), 2 had no start entity by name. The 3 that differ are CWQ, a
  path that now returns nothing and two whose middle hop crossed the 200 limit. **Labels recomputed by id: 100% agree
  in BioCQ, GrailQA, MetaQA and WebQSP, 98.1% in CWQ (1 row).** So A's labels are right, and its 98 name-only
  disagreements in 3.3 were id matches, as inferred.
- **3.7 B's queries reproduce on the graph.** Of 300 sampled plain queries, **292 return exactly the ids B logged
  (97.3%)**. Of the 8 others, 3 are KORA's own cap (it keeps 500 ids; the graph holds 592–1,594), and 5 return more
  ids than were logged (1 vs 4, 1 vs 2, 1 vs 3): a candidate set B under-records. Small, but those states' labels were
  computed on fewer ids than the query returns.
- **3.8 B's runs are complete.** Every one of the 5,000 base questions has a harness row; 3 errored, and every
  question but one errored GrailQA question has states in B (median 6–13 per question). Statuses: 4,184 solved,
  717 idle, 96 timed out (the 120 s clock). The check "states logged vs judge calls" is not a completeness test: a
  judge call scores several candidates at once (7–9 states per call), so it cannot show a missing state.
- **3.9 Near-duplicates exist that exact text missed:** the same question reworded in train and in a held-out set,
  e.g. *"where did frank baum die"* (test) vs *"where did l frank baum die"* (train, cosine 1.0); *"which cricket series
  has team two of england cricket?"* (holdout) vs *"what cricket series ..."* (train). At cosine ≥ 0.9: validation 1,
  holdout 1, test 8 (of 17,657); most pairs at 0.8–0.9 are template siblings with different entities (MetaQA, BioCQ),
  not duplicates. To do in cleaning: list every pair ≥ 0.85, read them, drop the train questions that are the same
  question.


## 4. Cleaning and pre-processing

### 4.1 Cleaning
What the model will read is a question, a path and a candidate; the target is the label. Cleaning keeps only rows that
have all of them, correctly, on train questions, and that a KGQA system could actually produce. Every drop is counted
by its reason (a row is counted under its first reason, in this order):

| dataset | reason | why |
|---|---|---|
| A, B | not a train question | validation, holdout, retired or test-text questions are never trained on |
| A | no named candidate | nothing to show ("(N items)" then "(empty)") |
| A | glued anchors | the path starts from all the question's entities at once and states a false fact |
| A | returns to its start | the next hop walks straight back to where it began (a repeated line); no system does this (ToG removes the relation it arrived by) |
| A | lucky | a yes whose path shares no relation with the gold path: right by overlap, not by the path |
| B | sub-question state | the oracle only knows the whole question's gold |
| B | no usable path | a root state whose path cannot be read from KORA's state text |
| B | no nameable candidate | no candidate the text could name |
| A, B | duplicate | the same state twice for a question |

Kept on purpose: same-relation paths that reach *new* nodes (a valid 2-hop pattern), the 837 B states KORA cut at
500 candidate ids, questions whose rows are all no, root states, candidates shown as raw ids (as live).

In [ ]:
DIRECTION = {">": "out", "<": "in", "out": "out", "in": "in"}
EDGE = re.compile(r"^(.*?) -\[(.*?)\]-> \{(.*)\} \((\d+)\)\s*$")

def glued(r):
    if len(r.evidence) == 0 or len(r.directions) == 0:     # lists, or arrays when read back from Parquet
        return False
    first = r.evidence[0].split(" | ")
    side = first[0] if DIRECTION[r.directions[0]] == "out" else first[-1]
    names = {norm(v) for v in topic(r.qid).values()}
    return sum(1 for x in side.split(", ") if norm(x) in names) >= 2

returns = lambda ev: len(ev) != len(set(ev))

def reason_A(r):
    if r.qid not in train:           return "not a train question"
    if len(r.candidate) == 0:        return "no named candidate"
    if glued(r):                     return "glued anchors"
    if returns(r.evidence):          return "returns to its start"
    if r.lucky:                      return "lucky"
    return None

ID_LIKE = re.compile(r"^([mg]\.[0-9a-z_]+|[A-Za-z]+KG:\S+|KQAPro:\S+)$")
def usable_path(r):
    return (isinstance(r.query, str) and r.query.strip() != "") or any(EDGE.match(l) for l in b_evidence(r.state))

def named_by_kora(r):
    return r.bench == "metaqa" or any(n and not ID_LIKE.match(str(n)) for n in (r.names or []))

def reason_B(r):
    if r.qid not in train:           return "not a train question"
    if r.is_subquestion:             return "sub-question state"
    if not usable_path(r):           return "no usable path"
    return None

A["dropped_for"] = [reason_A(r) for r in A.itertuples()]
B["dropped_for"] = [reason_B(r) for r in B.itertuples()]

# Candidates KORA left unnamed: ask the graph for their names (first 30 ids, what the text shows).
ask = B[B["dropped_for"].isna() & ~pd.Series([named_by_kora(r) for r in B.itertuples()], index=B.index)]
def graph_named(item):
    k, bench, ids = item
    try:
        nm = names(bench, [str(i) for i in ids][:30])
        return k, any(v and not ID_LIKE.match(str(v)) for v in nm.values())
    except Exception as e:
        return k, outcome_of(e)
looked = dict(run_parallel(graph_named, [(k, r.bench, r.ids) for k, r in ask.iterrows()], "names") if len(ask) else [])
for k, ok in looked.items():
    if ok is not True:
        B.loc[k, "dropped_for"] = "no nameable candidate" if ok is False else f"no nameable candidate ({ok})"
print(f"B states whose candidates KORA did not name: {len(ask)}; named by the graph: {sum(v is True for v in looked.values())}")

# Duplicates among what is left: the same state twice for a question.
dA = A[A["dropped_for"].isna()].duplicated(["qid", "text"]);                   A.loc[dA[dA].index, "dropped_for"] = "duplicate"
keyB = B.goal.astype(str) + "\u0001" + B.state.astype(str) + "\u0001" + B.ids.map(lambda x: json.dumps(list(x or [])))
dB = keyB[B["dropped_for"].isna()].duplicated();                                B.loc[dB[dB].index, "dropped_for"] = "duplicate"

counts = pd.DataFrame({"A": A["dropped_for"].fillna("kept").value_counts(), "B": B["dropped_for"].fillna("kept").value_counts()}).fillna(0).astype(int)
order = ["not a train question", "no named candidate", "glued anchors", "returns to its start", "lucky",
         "sub-question state", "no usable path", "no nameable candidate", "duplicate", "kept"]
counts = counts.reindex([k for k in order if k in counts.index] + [k for k in counts.index if k not in order])
counts.loc["rows before"] = [len(A), len(B)]
counts

In [ ]:
# What is left, per set.
A_clean = A[A["dropped_for"].isna()].copy(); B_clean = B[B["dropped_for"].isna()].copy()
def per_set(df):
    return df.groupby("bench").agg(rows=("label", "size"), questions=("qid", "nunique"), yes=("label", "mean"))
summary = pd.concat({"A": per_set(A_clean), "B": per_set(B_clean)}, axis=1).reindex(ALL)
summary.loc["total"] = [len(A_clean), A_clean.qid.nunique(), A_clean.label.mean(), len(B_clean), B_clean.qid.nunique(), B_clean.label.mean()]
summary.round(3)

In [ ]:
# Save the cleaned rows (still in their source format; rendering is 4.2).
os.makedirs(f"{V}/cleaned", exist_ok=True)
a_cols = ["qid", "bench", "source", "question", "evidence", "candidate", "n_candidate", "relations", "directions",
          "constraints", "constraint_lines", "label", "recall", "precision", "text"]
b_cols = ["qid", "bench", "site", "task", "goal", "is_subquestion", "state", "query", "ids", "names", "n_got", "n_gold",
          "label", "recall", "precision", "names_truncated", "ts", "cand"]
A_out = A_clean[a_cols].reset_index(drop=True)
B_out = B_clean[b_cols].reset_index(drop=True)
B_out["query"] = [q if isinstance(q, str) else None for q in B_out["query"]]
B_out["ids"] = B_out["ids"].map(lambda x: [str(i) for i in (x or [])]); B_out["names"] = B_out["names"].map(lambda x: [str(n) for n in (x or [])])
B_out["names_truncated"] = B_out["names_truncated"].fillna(False).astype(bool)
A_out.to_parquet(f"{V}/cleaned/A.parquet", index=False); B_out.to_parquet(f"{V}/cleaned/B.parquet", index=False)
print({f: f"{os.path.getsize(f'{V}/cleaned/{f}') / 1e6:.1f} MB" for f in sorted(os.listdir(f"{V}/cleaned"))})

### 4.1b Post-cleaning audit
Read back from the saved files: every check must be 0.

In [ ]:
A_chk, B_chk = pd.read_parquet(f"{V}/cleaned/A.parquet"), pd.read_parquet(f"{V}/cleaned/B.parquet")
lucky_left = A.loc[A["dropped_for"].isna(), "lucky"].sum()
checks = [
    ("A: rows not on a train question", int((~A_chk.qid.isin(train)).sum())),
    ("A: rows with no named candidate", int((A_chk.candidate.map(len) == 0).sum())),
    ("A: glued anchors", int(sum(glued(r) for r in A_chk.itertuples()))),
    ("A: paths returning to their start", int(A_chk.evidence.map(lambda e: returns(list(e))).sum())),
    ("A: lucky rows", int(lucky_left)),
    ("A: duplicate (question, text)", int(A_chk.duplicated(["qid", "text"]).sum())),
    ("B: rows not on a train question", int((~B_chk.qid.isin(train)).sum())),
    ("B: sub-question states", int(B_chk.is_subquestion.sum())),
    ("B: no usable path", int(sum(not usable_path(r) for r in B_chk.itertuples()))),
    ("B: duplicate states", int((B_chk.goal + B_chk.state + B_chk.ids.map(lambda x: json.dumps(list(x)))).duplicated().sum())),
    ("A, B: label not 0/1", int((~A_chk.label.isin([0, 1])).sum() + (~B_chk.label.isin([0, 1])).sum())),
    ("A, B: rows on validation, holdout, retired or test questions",
     int(A_chk.qid.isin(validation | holdout | retired | in_test).sum() + B_chk.qid.isin(validation | holdout | retired | in_test).sum())),
]
pd.DataFrame(checks, columns=["check", "count (must be 0)"])

**Cleaning result**
- **A: 118,527 → 84,881 rows** on 5,531 questions (16.4% yes). Dropped, by first reason: glued anchors 17,604,
  returning to the start 12,269, no named candidate 2,817, lucky 808, duplicates 148 (lucky and returning rows that
  also had an earlier reason are counted there). KQA Pro keeps 793 of its 1,000 questions and BioCQ 896 of 957: some
  multi-entity questions had only glued paths; the fixed generator (each path from one entity) can refill them.
- **B: 54,832 → 38,134 states** on 4,800 questions (23.9% yes). Dropped: sub-question states 13,789, rows on
  held-out questions 1,756, duplicates 1,092, no nameable candidate 61 (of 75 states whose candidates KORA had not
  named, the graph named 14). Every root state's path parses.
- Every post-cleaning check is 0. Saved: `cleaned/A.parquet` (42 MB), `cleaned/B.parquet` (9.5 MB).

### 4.2 Pre-processing

Every row becomes the pair the model reads, QNLI-style (the format ModernBERT's own starter script for custom data
uses): **`question`** and **`evidence`** (one triple per line, intermediate sets as `?x1`, constraint lines, then the
candidate with its count), plus **`label`**. Both come from one module, `render.py` (`estimator/render.py`), the live judge server's own code: the server imports the same file, so a
training row and a live request of the same state are the same text. Every pair is validated before it is kept.

- **B** (KORA states): `from_kora` renders from KORA's executed query and candidate ids (a root state from KORA's
  path text), with names from the graph, exactly as the server does.
- **A** (generated paths): `from_path` renders from the stored hops (relation, direction), the start entity, the
  constraints and the candidate, through the same `compose` as KORA's queries.

In [ ]:
import importlib, sys
from estimator import render
from estimator.render import from_kora, from_path, validate, joined, ids_in, ids_to_name, HEAD

# Names from the graph for every id a B state shows (candidates, query constants, path ids), as the server does.
need = collections.defaultdict(set)
for r in B_out.itertuples():
    ids = list(r.ids)
    for i in ids[:30] + ids_in(r.query) + (ids_to_name(r.state, ids) if not isinstance(r.query, str) else []):
        if re.fullmatch(r"[mg]\.[0-9a-z_]+", i):
            need["freebase"].add(i)
        elif "KG:" in i:
            need["biocq"].add(i)
batches = [(g, sorted(v)[k:k + 200]) for g, v in need.items() for k in range(0, len(v), 200)]
def name_batch(item):
    g, ids = item
    try:
        return names("webqsp" if g == "freebase" else "biocq", ids)
    except Exception as e:
        return {"__error__": outcome_of(e)}
NAMES = {}
for part in run_parallel(name_batch, batches, "naming"):
    NAMES.update({k: v for k, v in part.items() if k != "__error__" and v})
print({g: len(v) for g, v in need.items()}, "ids to name;", len(NAMES), "named;",
      sum(1 for g, v in need.items() for i in v if i not in NAMES), "left as ids (unnamed in the graph, as live)")

**Parity with the live server.** The DAgger runs logged, for every state, what KORA sent and the text the server
wrote. Re-rendering a sample of them with `render.py` must give the same text.

In [ ]:
live = LIVE[LIVE.extra.map(lambda e: bool(e))].sample(500, random_state=0)
extra_need = collections.defaultdict(set)
for r in live.itertuples():
    e = r.extra
    chain_ids = [str(i) for link in (e.get("chain") or []) for i in list(link.get("ids") or [])[:30] + ids_in(link.get("query"))]
    for i in [str(x) for x in (e.get("ids") or [])][:30] + ids_in(e.get("query")) + ids_to_name(r.state or "", e.get("ids") or []) + chain_ids:
        if re.fullmatch(r"[mg]\.[0-9a-z_]+", i) and i not in NAMES: extra_need["freebase"].add(i)
        elif "KG:" in i and i not in NAMES: extra_need["biocq"].add(i)
for part in run_parallel(name_batch, [(g, sorted(v)[k:k + 200]) for g, v in extra_need.items() for k in range(0, len(v), 200)], "naming"):
    NAMES.update({k: v for k, v in part.items() if k != "__error__" and v})
same = []
for r in live.itertuples():
    e = r.extra
    q, ev = from_kora(r.question, r.state or "", e.get("ids") or [], e.get("n"), e.get("query"), NAMES, e.get("chain") or [])
    same.append(joined(q, ev) == r.text)
live = live.assign(same=same)
print(f"re-rendered = logged live text: {np.mean(same):.1%} of {len(same)}")
for r in live[~live.same].head(2).itertuples():
    print("--- live\n" + r.text[:600]); q, ev = from_kora(r.question, r.state or "", r.extra.get("ids") or [], r.extra.get("n"), r.extra.get("query"), NAMES, r.extra.get("chain") or []); print("--- ours\n" + joined(q, ev)[:600])

In [ ]:
# B: render every state.
def render_B(r):
    return from_kora(r.goal, r.state, list(r.ids), r.n_got, r.query, NAMES)
pairs = [render_B(r) for r in B_out.itertuples()]
B_pre = B_out.assign(question=[q for q, _ in pairs], evidence=[e for _, e in pairs])
B_pre["problems"] = [validate(q, e) for q, e in pairs]
print("B states failing validation:", collections.Counter(p for ps in B_pre.problems for p in ps) or "none")

In [ ]:
# A: the path's start, hops and constraints, read from the stored evidence lines in order.
def a_parts(r):
    rels, dirs = list(r.relations), list(r.directions)
    pending = [tuple(x.split(" = ", 1)) for x in r.constraint_lines]
    hop, start, cons = 0, None, []
    for line in r.evidence:
        s, rel, o = (line.split(" | ") + ["", "", ""])[:3]
        if pending and (rel, o) == pending[0]:
            cons.append((hop, rel, o)); pending.pop(0)
        elif hop < len(rels) and rel == rels[hop]:
            if hop == 0:
                side = s if render.DIRECTION[dirs[0]] == "out" else o
                start = None if side.startswith("?x") else side
            hop += 1
        else:
            return None, "path lines do not match its hops"
    if hop != len(rels) or pending:
        return None, "path lines do not match its hops"
    if any(" & " in x or " / " in x for x in rels):
        return None, "a joined hop (no single relation)"
    return (start, list(zip(rels, dirs)), [c for c in cons if c[1] != "type"]), None

parts = [a_parts(r) for r in A_out.itertuples()]
A_pre = A_out.assign(problem=[why for _, why in parts])
ok = A_pre.problem.isna()
pairs = [from_path(r.question, *pt, list(r.candidate), r.n_candidate) if pt else (None, None)
         for r, (pt, _) in zip(A_out.itertuples(), parts)]
A_pre["question"] = [q if q is not None else r.question for (q, _), r in zip(pairs, A_out.itertuples())]
A_pre["evidence"] = [e for _, e in pairs]
A_pre["problems"] = [validate(q, e) if e else [why] for (q, e), why in zip(pairs, A_pre.problem)]
print("A rows not renderable:", A_pre.problem.value_counts().to_dict() or "none")
print("A rows failing validation:", collections.Counter(p for ps in A_pre.problems for p in ps) or "none")

In [ ]:
# Does the rendered text look like live text now? (the shapes of section 1.3)
def shape_pair(q, ev):
    return shape(joined(q, ev))
keepA, keepB = A_pre[A_pre.problems.map(len) == 0], B_pre[B_pre.problems.map(len) == 0]
pd.DataFrame({"A before (generator's text)": pd.DataFrame([shape(t) for t in A_out.text]).mean(),
              "A rendered": pd.DataFrame([shape_pair(q, e) for q, e in zip(keepA.question, keepA.evidence)]).mean(),
              "B rendered": pd.DataFrame([shape_pair(q, e) for q, e in zip(keepB.question, keepB.evidence)]).mean(),
              "live": pd.DataFrame([shape(t) for t in LIVE.text]).mean()}).round(3)

In [ ]:
# Identical text with different labels, after rendering: within A, within B, and across them.
def conflicts(df):
    g = df.groupby(["question", "evidence"]).label.nunique()
    return int((g > 1).sum()), int(df.duplicated(["qid", "question", "evidence"]).sum())
both = pd.concat([keepA[["qid", "question", "evidence", "label"]], keepB[["qid", "question", "evidence", "label"]]])
pd.DataFrame({k: dict(zip(["identical text, both labels", "duplicate (question, text)"], conflicts(d)))
              for k, d in (("A", keepA), ("B", keepB), ("A and B together", both))})

In [ ]:
# Resolve: a text that carries both labels cannot be learned; drop those rows. Remove duplicates. Then save.
def finish(df):
    bad = df.groupby(["question", "evidence"]).label.transform("nunique") > 1
    out = df[~bad].drop_duplicates(["qid", "question", "evidence"]).reset_index(drop=True)
    return out, int(bad.sum())
# A's candidate sets the generator stopped at 200 nodes: a "yes" there cannot be re-checked from what is stored
# (A keeps names, not ids), so those rows are dropped; the "no" rows stay.
cut_yes = (keepA.n_candidate == 200) & (keepA.label == 1)     # exactly the generator's limit
print(f"A rows dropped: candidate cut at the generator's 200 limit and labelled yes: {int(cut_yes.sum())}")
keepA = keepA[~cut_yes]
A_fin, a_bad = finish(keepA); B_fin, b_bad = finish(keepB)
# The model reads `question` and `evidence` and learns `label`; everything else is metadata in one `meta` column
# (filtering, grouped cross-validation, per-set results), never passed to the model.
A_ds = pd.DataFrame({"question": A_fin.question, "evidence": A_fin.evidence, "label": A_fin.label.astype(int),
                     "meta": [{"qid": q, "bench": b, "source": f"generated:{s}", "site": "leaf"} for q, b, s in zip(A_fin.qid, A_fin.bench, A_fin.source)]})
B_ds = pd.DataFrame({"question": B_fin.question, "evidence": B_fin.evidence, "label": B_fin.label.astype(int),
                     "meta": [{"qid": q, "bench": b, "source": "kora", "site": st, "ts": float(t), "cand": str(c or "")}
                              for q, b, st, t, c in zip(B_fin.qid, B_fin.bench, B_fin.site, B_fin.ts, B_fin.cand)]})
for name, df in (("A", A_ds), ("B", B_ds)):
    df.insert(0, "idx", range(len(df)))
    os.makedirs(f"{V}/rendered", exist_ok=True)
    df.to_parquet(f"{V}/rendered/{name}.parquet", index=False)                       # meta: a struct column
    df.assign(meta=df.meta.map(json.dumps)).to_csv(f"{V}/rendered/{name}.csv", index=False)   # meta: JSON text
print(f"dropped for carrying both labels: A {a_bad}, B {b_bad}")
pd.DataFrame({n: {"rows": len(d), "questions": d.meta.map(lambda m: m["qid"]).nunique(), "yes": round(d.label.mean(), 3)} for n, d in (("A", A_ds), ("B", B_ds))})

In [ ]:
# Five rows of each, as stored: the three model columns and the metadata.
with pd.option_context("display.max_colwidth", 160):
    display(A_ds.sample(5, random_state=1))
    display(B_ds.sample(5, random_state=1))

In [ ]:
# One row of each in full.
for name, df in (("A", A_ds), ("B", B_ds)):
    r = df.sample(1, random_state=3).iloc[0]
    print(f"===== {name} | label {r.label} | meta {r.meta}\n[question]\n{r.question}\n[evidence]\n{r.evidence}\n")

### 4.2b Is anything cut?
Three places can cut a state: the **display** (at most 30 names per list, then `... (+N more)` with the true count:
by design, as live), the **collection** (KORA keeps at most 500 candidate ids; the generator follows at most 200
nodes per hop), and the **model's 1,024-token limit** (which would cut the end of the evidence: the candidate).
Read back from the saved files.

**No input is truncated.** The maximum length is set to **7,999 tokens** for every model (Ettin's
context length, all sizes; ModernBERT's is 8,192), so every row fits as rendered; batches are grouped by length; results
are reported by length bucket. The 30-name display cap stays (the count `(N items)` is always the true one); showing
full lists is a later option to test.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(f"{DATA_DIR}/kora/models/ModernBERT-large")
rows = []
for name in ("A", "B"):
    d = pd.read_parquet(f"{V}/rendered/{name}.parquet")
    lens = np.array([len(x) for x in tok(d.question.tolist(), d.evidence.tolist())["input_ids"]])
    n_shown = d.evidence.str.extract(r"CANDIDATE ANSWER \((\d+) items\)")[0].astype(int)
    rows.append({"dataset": name, "rows": len(d),
                 "fail validation": int(sum(bool(validate(q, e)) for q, e in zip(d.question, d.evidence))),
                 "tokens median": int(np.median(lens)), "tokens p99": int(np.percentile(lens, 99)), "tokens max": int(lens.max()),
                 "over 512 tokens": int((lens > 512).sum()), "over 1,024 tokens": int((lens > 1024).sum()),
                 "over 7,999 tokens (the models' limit: would be cut)": int((lens > 7999).sum()),
                 "a list shortened (+N more)": int(d.evidence.str.contains(r"\(\+\d+ more\)").sum()),
                 "candidate at a collection limit": int(((n_shown == 200) if name == "A" else (n_shown >= 500)).sum()),
                 "  of which labelled yes": int(d.label[(n_shown == 200) if name == "A" else (n_shown >= 500)].sum())})
pd.DataFrame(rows).set_index("dataset").T

In [ ]:
# The longest row, to see what a long state looks like.
d = pd.read_parquet(f"{V}/rendered/A.parquet")
k = int(np.argmax([len(x) for x in tok(d.question.tolist(), d.evidence.tolist())["input_ids"]]))
print(d.question[k]); print(d.evidence[k][:1500])

**Pre-processing result**
- **The renderer is the live server's:** re-rendering 500 logged live states with `render.py` gives the logged text
  **100%** of the time, so training text and inference text are the same function.
- **B:** 38,134 states rendered; 13 fail validation (the count below the names shown), 961 render identically to
  another state of the same question (removed), 5 texts carry both labels (10 rows dropped): **37,155 rows, 22.2% yes**.
- **A:** 84,881 rows rendered from their hops through the same `compose`; 48 could not be read back into hops and
  52 fail validation, and 45 yes rows whose candidate set the generator stopped at its 200 limit cannot be re-checked (all dropped): **84,735 rows, 16.3% yes**.
- **Format now matches live:** A's text went from 826 to 581 characters and from 56% to 27% "list as a subject";
  duplicated lines 0. What still differs is content, not format: A's paths are longer (2.4 evidence lines against
  1.3 live), because generated paths are often 2-hop and carry constraint lines, while KORA's decomposition makes most
  live states one triple.
- **Across A and B** the same question and text appears 10,234 times and 39 texts carry both labels: when A and B
  are mixed for training (the DAgger rounds), they must be de-duplicated and those texts dropped.
- Saved: `rendered/A.parquet` (+ `.csv`), `rendered/B.parquet` (+ `.csv`); columns `idx`, `question`,
  `evidence`, `label` (what the model sees) and `meta` (`qid`, `bench`, `source`, `site`; never passed to the model).
- **Nothing is cut:** every row is under 7,999 tokens (longest 7,010), the models' shared limit; every row passes validation.

### 4.3 Validation states
To score a judge offline, validation needs **states**: one KORA run on the 666 validation questions (arm E KORA,
`kora_oracle_nomem`: the gold decides the root, memory off; KORA's own leaf judge), every judged state logged by the
harvest logger with all its candidate ids, including the query agent's own leaf judgements
(`KORA_HARVEST_AT_AGENT=1`). Each state goes through exactly what B went through:
sub-question states dropped, the oracle's label against the gold, `render.py`, validation, duplicates removed. One
validation set serves every judge, trained on A or on B.

In [ ]:
VAL = pd.DataFrame(jsonl_dir := [json.loads(l) for f in sorted(glob.glob(f"{V}/runs/val_oracle2/harvest/*.jsonl")) for l in open(f) if l.strip()])
VAL["qid"] = VAL.qid.astype(str)
VAL["goal"] = VAL.goal.fillna("").astype(str)
VAL["query"] = VAL["query"] if "query" in VAL else None
VAL["is_subquestion"] = [norm(g) != norm(QUESTION.get(q, {}).get("question", "")) for g, q in zip(VAL.goal, VAL.qid)]
reason = []
for r in VAL.itertuples():
    if r.qid not in validation:               reason.append("not a validation question")
    elif r.is_subquestion:                    reason.append("sub-question state")
    elif len(r.ids or []) == 0:               reason.append("no candidate")
    else:                                     reason.append(None)
VAL["dropped_for"] = reason
cut = VAL["query"].map(lambda q: isinstance(q, str) and len(q) >= 600)
print(f"logged states: {len(VAL)} on {VAL.qid.nunique()} questions | by site: {VAL.site.value_counts().to_dict()}")
print("dropped:", VAL.dropped_for.value_counts().to_dict(), f"| queries at the logger's 600-character cut: {int(cut.sum())}")

In [ ]:
# Label (the oracle's rule against the gold), name, render, validate: as for B.
keep = VAL[VAL.dropped_for.isna()].copy()
need = collections.defaultdict(set)
for r in keep.itertuples():
    ids = [str(i) for i in r.ids]
    q = r.query if isinstance(r.query, str) else None
    for i in ids[:30] + ids_in(q) + (ids_to_name(r.state, ids) if not q else []):
        if re.fullmatch(r"[mg]\.[0-9a-z_]+", i) and i not in NAMES: need["freebase"].add(i)
        elif "KG:" in i and i not in NAMES: need["biocq"].add(i)
for part in run_parallel(name_batch, [(g, sorted(v)[k:k + 200]) for g, v in need.items() for k in range(0, len(v), 200)], "naming"):
    NAMES.update({k: v for k, v in part.items() if k != "__error__" and v})
# The label: ids against gold ids, and names against gold names (GrailQA's gold has names only): the oracle's rule.
def cand_names(ids, logged):
    return [logged[k] if k < len(logged) and logged[k] else NAMES.get(i, i) for k, i in enumerate(ids)]
keep["label"] = [oracle([str(i) for i in ids], cand_names([str(i) for i in ids], list(nm or [])), q)[0]
                 for ids, nm, q in zip(keep.ids, keep.names, keep.qid)]
pairs = [from_kora(r.goal, r.state, [str(i) for i in r.ids], len(r.ids), r.query, NAMES) for r in keep.itertuples()]
keep["question"] = [q for q, _ in pairs]; keep["evidence"] = [e for _, e in pairs]
keep["problems"] = [validate(q, e) for q, e in pairs]
print("failing validation:", collections.Counter(x for ps in keep.problems for x in ps) or "none")
keep = keep[keep.problems.map(len) == 0]
both = keep.groupby(["question", "evidence"]).label.transform("nunique") > 1
VAL_ds = keep[~both].drop_duplicates(["qid", "question", "evidence"])
VAL_ds = pd.DataFrame({"idx": range(len(VAL_ds)), "question": VAL_ds.question.values, "evidence": VAL_ds.evidence.values,
                       "label": VAL_ds.label.astype(int).values,
                       "meta": [{"qid": q, "bench": BENCH[q], "source": "kora-val-oracle", "site": s, "ts": float(t), "cand": str(c or "")}
                                for q, s, t, c in zip(VAL_ds.qid, VAL_ds.site, VAL_ds.ts, VAL_ds.cand)]})
VAL_ds.to_parquet(f"{V}/validation.parquet", index=False)
VAL_ds.assign(meta=VAL_ds.meta.map(json.dumps)).to_csv(f"{V}/validation.csv", index=False)
print(f"texts with both labels dropped: {int(both.sum())}; duplicates removed: {len(keep[~both]) - len(VAL_ds)}")
vb = VAL_ds.assign(bench=VAL_ds.meta.map(lambda m: m["bench"]), site=VAL_ds.meta.map(lambda m: m["site"]))
t = vb.groupby("bench").agg(states=("label", "size"), questions=("meta", lambda m: len({x["qid"] for x in m})), yes=("label", "mean"))
t.loc["total"] = [len(vb), vb.meta.map(lambda m: m["qid"]).nunique(), vb.label.mean()]
display(t.round(3)); print("by site:", vb.groupby("site").label.agg(["size", "mean"]).round(3).to_dict())

In [ ]:
# Tokens: within the 7,999 limit?
lens = np.array([len(x) for x in tok(VAL_ds.question.tolist(), VAL_ds.evidence.tolist())["input_ids"]])
print(f"tokens: median {int(np.median(lens))}, p99 {int(np.percentile(lens, 99))}, max {lens.max()}, over 7,999: {int((lens > 7999).sum())}")

**Validation states result**
- 8,469 states logged on all 666 questions (7,841 leaf, 628 root); 2,076 claim-style sub-question states dropped (as
  for B); 1 fails validation; 767 duplicates removed; no text with both labels; no query at the logger's cut.
- **5,625 validation states, 19.8% yes** (B: 22.2%): leaves 5,207 (16.1% yes), roots 418 (67.0% yes). Per set:
  BioCQ 916 (19.9%), CWQ 1,549 (13.6%), GrailQA 1,257 (18.3%), MetaQA 814 (32.8%), WebQSP 1,089 (20.8%).
- Tokens: median 84, p99 526, max 1,132: nothing near the 7,999 limit.
- Saved `validation.parquet` (+ `.csv`), the same columns as train.

### 4.4 The page: the schema the query agent saw
For the schema experiment (§6.1) every state gets one more column, **`page`**: the schema of the subgraph the query
agent was looking at when it wrote the state's query. It is not part of the baseline input (§5); training decides
whether to use it.

**Where it comes from.** The harness logs every model call's full prompt: B's runs (the training questions) and the validation run both
have them. A state is matched to the **last call for its goal before it was judged** (`meta.ts`):

| state | its page |
|---|---|
| round 1, round 2 | that round's page (round 2's adds the entities round 1 found, so it differs) |
| round 3 (the neighbourhood search, no page) | the neighbourhood list the agent saw |
| root | the last page of the whole-question call |

**What is stripped** (written for the LLM, not schema): the `TRIED:` notes (the agent's own history), the
instructions ("-- go through it, do not return it", "the answer is at the far end, never the middle", "a query may use
any of them past the anchor"), and three blocks: `HOW THIS GRAPH WORKS` (how to write ids in a query), `BOUND SETS`
(sets earlier tasks bound) and the catalogue shown when an entity has almost no relations (relations of other entities).
Kept: every relation line (direction, count, target types, examples, "?x is a MEDIATOR"), the mediator routes, and on
MetaQA and BioCQ the whole graph's relation list.

**Fit**: the page is kept whole when question + evidence + page fit in 7,999 tokens. Otherwise
the relation lines are ranked by closeness to the goal with KORA's own embedder (bge-small-en-v1.5, the [CLS] vector,
cosine, relation names made readable as KORA does) and the lowest are dropped until it fits; **lines with a relation of
the state's own path are always kept**; a shortened block ends with `(+N lower-ranked not shown)`.

B was collected with **memory on** (arm `kora_oracle`), validation with memory off: memory text is not in the page, and
the page order memory sets is replaced by the embedder's ranking whenever a page is trimmed.

In [ ]:
# Every page and neighbourhood list the query agent was given, from the runs' logged prompts.
import bisect, torch
from transformers import AutoModel

RESULTS = f"{DATA_DIR}/results/eval"
TRACES = {"B": [f"{RESULTS}/train-kora_oracle-qwen3-30b-a3b-{s}/qwen3-30b-a3b/{s}-train/kora_oracle/traces" for s in SETS],
          "validation": [f"{RESULTS}/sj2-val-oracle2/qwen3-30b-a3b/{s}-val/kora_oracle_nomem/traces" for s in SETS]}

def between(text, start, end):
    return text.split(start, 1)[1].split(end, 1)[0] if start in text else ""

def logged_calls(dirs):
    """(question id, goal) -> [(time, 'page' | 'list', text)] in time order."""
    calls = collections.defaultdict(list)
    for d in dirs:
        for f in glob.glob(f"{d}/*/*.jsonl"):
            for line in open(f):
                c = json.loads(line); p = c["prompt"]
                if p.startswith("ORIGINAL QUESTION"):                                    # rounds 1 and 2: query.generate
                    kind, text = "page", between(p, "LOCAL SCHEMA\n", "\n\nSEEDS")
                elif p.startswith("TASK GOAL") and "WHAT SURROUNDS THE ENTITY" in p:     # round 3: query.explore
                    kind, text = "list", between(p, "its id in square brackets:\n", "\n\nCHOOSE FROM THIS LIST")
                else:
                    continue
                calls[(str(c["qid"]), norm(between(p, "TASK GOAL\n", "\n\nEXPECTED OUTPUT")))].append((c["ts"], kind, text))
    for v in calls.values():
        v.sort(key=lambda x: x[0])
    return calls

def page_of(calls, qid, ts, site):
    """The last page (or round-3 list) of the whole-question call before the state was judged, and its round."""
    v = calls.get((qid, norm(QUESTION[qid]["question"])), [])
    if site == "root":
        v = [x for x in v if x[1] == "page"]
    k = bisect.bisect_right([t for t, _, _ in v], ts)
    if k == 0:
        return "", "none"
    _, kind, text = v[k - 1]
    return text, ("root" if site == "root" else "3" if kind == "list" else str(min(sum(x[1] == "page" for x in v[:k]), 2)))

# What was written for the LLM, not schema.
DROP_BLOCKS = ("HOW THIS GRAPH WORKS:", "BOUND SETS", "THIS ENTITY HAS ALMOST NO RELATIONS")
REWRITE = [(re.compile(r"(;\s*|\s{2,})TRIED: [A-Z_]+"), ""),
           (re.compile(r" -- go through it, do not return it"), ""),
           (re.compile(r" \(see ROUTES\) but also reaches:"), ", also reaches:"),
           (re.compile(r" \(two steps, one fact -- the answer is at the far end, never the middle\)"), " (two steps)"),
           (re.compile(r"; a query may use any of them past the anchor\)"), ")")]
LLM_TEXT = ("TRIED", "go through it", "do not return", "HOW THIS GRAPH", "BOUND SETS", "ALMOST NO RELATIONS",
            "never the middle", "past the anchor")

def blocks_of(text):
    """A page as blocks: a header line and its indented lines."""
    out = []
    for line in text.splitlines():
        if line.strip():
            if line.startswith(" ") and out:
                out[-1][1].append(line)
            else:
                out.append([line, []])
    return out

def stripped(text):
    for pattern, new in REWRITE:
        text = pattern.sub(new, text)
    return [[h, ls] for h, ls in blocks_of(text) if not h.startswith(DROP_BLOCKS)]

SCHEMA_HEAD = "SCHEMA (the relations the query agent saw around the anchors)"
LIST_HEAD = "NEIGHBOURHOOD (the facts the query agent saw around the anchor)"

def render_page(blocks, rnd):
    return "\n".join([LIST_HEAD if rnd == "3" else SCHEMA_HEAD] + [x for h, ls in blocks for x in [h] + ls]) if blocks else ""

In [ ]:
# The ranking used to trim: KORA's embedder (bge-small-en-v1.5, the [CLS] vector, normalised; tools/embed.py).
EMB = f"{DATA_DIR}/kora/models/bge-small-en-v1.5"
emb_tok, emb = AutoTokenizer.from_pretrained(EMB), AutoModel.from_pretrained(EMB).eval()

def embed(texts):
    out = []
    for k in range(0, len(texts), 256):
        b = emb_tok(texts[k:k + 256], padding=True, truncation=True, max_length=512, return_tensors="pt")
        with torch.no_grad():
            out.append(torch.nn.functional.normalize(emb(**b).last_hidden_state[:, 0], dim=-1).numpy())
    return np.concatenate(out) if out else np.zeros((0, 384))

readable = lambda r: r.replace("_", " ").replace("/", " ").replace(".", " ").lower()      # as KORA's tools/embed.py
REL = [re.compile(r"^\s+(\S+): \S+ -> "),                              # whole graph:  rel: from -> to
       re.compile(r"\s(\S+) \?mid \. \?mid (\S+) \?answer"),          # route:        a r1 ?mid . ?mid r2 ?answer
       re.compile(r"^\s+\?x (\S+) "), re.compile(r"\s(\S+) \?x\b")]    # pattern:      ?x r a  |  a r ?x

def relations_in(line):
    for rx in REL:
        m = rx.search(line)
        if m:
            return list(m.groups())
    return []

# Check: on validation pages (memory off, so KORA ordered each anchor's lines by this similarity, then by count),
# our similarity must give back the logged order.
calls_val = logged_calls(TRACES["validation"])
agree = checked = 0
for (q, goal), v in random.Random(0).sample(sorted(calls_val.items()), 150):
    for h, ls in blocks_of(v[0][2]) if v[0][1] == "page" else []:
        if not h.startswith("PATTERNS FOR") or "closest to the goal are described" in h:
            continue                                        # anchor blocks KORA ordered in full
        lines = [(relations_in(l)[0], int(re.search(r"\((\d+)\)", l).group(1))) for l in ls if relations_in(l) and re.search(r"\((\d+)\)", l)]
        if len(lines) < 3:
            continue
        vec = embed([goal] + [readable(r) for r, _ in lines]); s = vec[1:] @ vec[0]
        checked += 1; agree += sorted(range(len(lines)), key=lambda k: (-round(float(s[k]), 5), -lines[k][1])) == list(range(len(lines)))
print(f"our ranking = KORA's logged order in {agree} of {checked} anchor blocks")

In [ ]:
# Fit: the page whole if the row fits in 7,999 tokens; else the lowest-ranked relation lines dropped, the path's kept.
LIMIT = 7999

def path_relations(evidence):
    return {p.split(" | ")[1].strip() for p in evidence.splitlines() if p.count(" | ") >= 2}

def fitted(question, evidence, blocks, rnd, g, vectors):
    page = render_page(blocks, rnd)
    n = len(tok(question, evidence + "\n\n" + page)["input_ids"])
    if n <= LIMIT:
        return page, 0
    keep = path_relations(evidence)
    lines = sorted(((b, i, max((float(vectors[readable(r)] @ g) for r in relations_in(l)), default=-1.0))
                    for b, (_, ls) in enumerate(blocks) for i, l in enumerate(ls) if not set(relations_in(l)) & keep),
                   key=lambda x: x[2])
    dropped, over = set(), n - LIMIT
    for b, i, _ in lines:                                   # a first cut from each line's own token count
        if over <= 0:
            break
        dropped.add((b, i)); over -= len(tok(blocks[b][1][i], add_special_tokens=False)["input_ids"]) + 1
    while True:                                             # then one more line at a time until it fits
        out = []
        for b, (h, ls) in enumerate(blocks):
            kept = [l for i, l in enumerate(ls) if (b, i) not in dropped]
            out.append([h, kept + ([f"  (+{len(ls) - len(kept)} lower-ranked not shown)"] if len(kept) < len(ls) else [])])
        page = render_page(out, rnd)
        if len(tok(question, evidence + "\n\n" + page)["input_ids"]) <= LIMIT or len(dropped) == len(lines):
            return page, len(dropped)
        dropped.add(next((b, i) for b, i, _ in lines if (b, i) not in dropped))

def with_pages(path, calls):
    """Add `page` (and meta round, lines dropped) to a saved dataset; the four model columns must not change."""
    ds = pd.read_parquet(path)
    before = ds[["idx", "question", "evidence", "label"]].copy()
    found = [page_of(calls, m["qid"], m["ts"], m["site"]) for m in ds.meta]
    blocks = [stripped(t) for t, _ in found]
    rounds = [r for _, r in found]
    whole = [render_page(b, r) for b, r in zip(blocks, rounds)]
    n = np.array([len(x) for x in tok(ds.question.tolist(), (ds.evidence + "\n\n" + pd.Series(whole)).tolist())["input_ids"]])
    over = np.flatnonzero(n > LIMIT)
    texts = sorted({readable(r) for k in over for _, ls in blocks[k] for l in ls for r in relations_in(l)})
    goals = sorted({ds.question[k] for k in over})
    vectors = dict(zip(texts + goals, embed(texts + goals)))
    pages, dropped = list(whole), [0] * len(ds)
    for k in over:
        pages[k], dropped[k] = fitted(ds.question[k], ds.evidence[k], blocks[k], rounds[k], vectors[ds.question[k]], vectors)
    ds["page"] = pages
    ds["meta"] = [{**m, "page_round": r, "page_dropped": int(d)} for m, r, d in zip(ds.meta, rounds, dropped)]
    assert ds[["idx", "question", "evidence", "label"]].equals(before)
    ds.to_parquet(path, index=False)
    ds.assign(meta=ds.meta.map(json.dumps)).to_csv(path.replace(".parquet", ".csv"), index=False)
    return ds, n

calls_B = logged_calls(TRACES["B"])
PAGED = {"B": with_pages(f"{V}/rendered/B.parquet", calls_B),
         "validation": with_pages(f"{V}/validation.parquet", calls_val)}

In [ ]:
# Checks and sizes, read back from the saved files.
rows = {}
for name, path in (("B", f"{V}/rendered/B.parquet"), ("validation", f"{V}/validation.parquet")):
    ds = pd.read_parquet(path); n_whole = PAGED[name][1]
    full = np.array([len(x) for x in tok(ds.question.tolist(), (ds.evidence + "\n\n" + ds.page).tolist())["input_ids"]])
    page_len = np.array([len(x) for x in tok(ds.page.tolist(), add_special_tokens=False)["input_ids"]])
    rnd = ds.meta.map(lambda m: m["page_round"])
    rows[name] = {"rows": len(ds), **{f"page from round {r}": int((rnd == r).sum()) for r in ("1", "2", "3", "root")},
                  "no page found (must be ~0)": int((rnd == "none").sum()),
                  "LLM-directed text left (must be 0)": int(sum(any(w in p for w in LLM_TEXT) for p in ds.page)),
                  "page tokens median": int(np.median(page_len)), "page tokens p90": int(np.percentile(page_len, 90)),
                  "rows over 7,999 with the whole page": int((n_whole > LIMIT).sum()),
                  "rows trimmed": int((ds.meta.map(lambda m: m["page_dropped"]) > 0).sum()),
                  "lines dropped per trimmed row, median": float(np.median([m["page_dropped"] for m in ds.meta if m["page_dropped"]] or [0])),
                  "rows over 7,999 after (must be 0)": int((full > LIMIT).sum()),
                  "row tokens with the page: median": int(np.median(full)), "p99": int(np.percentile(full, 99))}
pd.DataFrame(rows)

In [ ]:
# One state with its page, as the model will read it in §6.1.
ds = pd.read_parquet(f"{V}/rendered/B.parquet")
r = ds[ds.meta.map(lambda m: m["page_round"] == "2" and m["page_dropped"] == 0)].sample(1, random_state=4).iloc[0]
print(f"label {r.label} | {dict(r.meta)}\n[question]\n{r.question}\n[evidence]\n{r.evidence}\n\n{r.page[:2500]}")

**Page result:** every state has its page (no page found: 1 of 37,155 B rows, 0 of 5,625 validation states); no LLM-directed text is left. Median page 1,770 tokens (B) and 2,609 (validation); 1,370 B rows and 1,366 validation rows exceed 7,999 tokens with the whole page and are trimmed by rank (median 22 and 111 relation lines dropped); none exceeds 7,999 after. Our ranking reproduces KORA's logged order in 153 of 156 anchor blocks.

### 4.4b Holdout states
The holdout's own KORA run (arm E, the gold deciding the root, memory off, `KORA_HARVEST_AT_AGENT=1`, call
traces on) over the 1,000 holdout questions of the five sets; KQA Pro's 200 are left out (no graph server, and no
judge has seen KQA Pro states). Built exactly as the validation states (4.3): whole-question states only, the oracle's
label from the gold, `render.py`, validation, texts with both labels dropped, duplicates removed; then each state's
page (4.4). Read once, in 6.3.

In [ ]:
# The holdout run's logged states, as in 4.3.
HOLD = pd.DataFrame([json.loads(l) for f in sorted(glob.glob(f"{V}/runs/hold_oracle/harvest/*.jsonl")) for l in open(f) if l.strip()])
HOLD["qid"] = HOLD.qid.astype(str)
HOLD["goal"] = HOLD.goal.fillna("").astype(str)
HOLD["query"] = HOLD["query"] if "query" in HOLD else None
HOLD["is_subquestion"] = [norm(g) != norm(QUESTION.get(q, {}).get("question", "")) for g, q in zip(HOLD.goal, HOLD.qid)]
HOLD["dropped_for"] = ["not a holdout question" if q not in holdout else "sub-question state" if sq else
                       "no candidate" if len(ids if ids is not None else []) == 0 else None
                       for q, sq, ids in zip(HOLD.qid, HOLD.is_subquestion, HOLD.ids)]
import sys; sys.path.insert(0, f"{DATA_DIR}/stopjudge/eval_overlay_dag")
from harness.metrics.reasoning import live_files                 # the live shard files only (sN.jsonl); sN.failed.jsonl are retried attempts
run_rows = [json.loads(l) for c in sorted(glob.glob(f"{RESULTS}/sj2-hold-oracle/qwen3-30b-a3b/*/kora_oracle_nomem"))
            for f in live_files(c) for l in open(f) if l.strip()]
ran = {str(r["qid"]) for r in run_rows if not r.get("error")}
print(f"logged states: {len(HOLD)} on {HOLD.qid.nunique()} questions | by site: {HOLD.site.value_counts().to_dict()}")
print("dropped:", HOLD.dropped_for.value_counts().to_dict())
print(f"holdout questions run cleanly: {len(ran)} of 1,000; errors by reason:",
      collections.Counter(str(r.get("error"))[:40] for r in run_rows if r.get("error")).most_common(3))

In [ ]:
# Label, name, render, validate (as 4.3), then save.
keep = HOLD[HOLD.dropped_for.isna()].copy()
need = collections.defaultdict(set)
for r in keep.itertuples():
    ids = [str(i) for i in r.ids]
    q = r.query if isinstance(r.query, str) else None
    for i in ids[:30] + ids_in(q) + (ids_to_name(r.state, ids) if not q else []):
        if re.fullmatch(r"[mg]\.[0-9a-z_]+", i) and i not in NAMES: need["freebase"].add(i)
        elif "KG:" in i and i not in NAMES: need["biocq"].add(i)
for part in run_parallel(name_batch, [(g, sorted(v)[k:k + 200]) for g, v in need.items() for k in range(0, len(v), 200)], "naming"):
    NAMES.update({k: v for k, v in part.items() if k != "__error__" and v})
keep["label"] = [oracle([str(i) for i in ids], cand_names([str(i) for i in ids], list(nm if nm is not None else [])), q)[0]
                 for ids, nm, q in zip(keep.ids, keep.names, keep.qid)]
pairs = [from_kora(r.goal, r.state, [str(i) for i in r.ids], len(r.ids), r.query, NAMES) for r in keep.itertuples()]
keep["question"] = [q for q, _ in pairs]; keep["evidence"] = [e for _, e in pairs]
keep["problems"] = [validate(q, e) for q, e in pairs]
print("failing validation:", collections.Counter(x for ps in keep.problems for x in ps) or "none")
keep = keep[keep.problems.map(len) == 0]
both = keep.groupby(["question", "evidence"]).label.transform("nunique") > 1
HOLD_ds = keep[~both].drop_duplicates(["qid", "question", "evidence"])
HOLD_ds = pd.DataFrame({"idx": range(len(HOLD_ds)), "question": HOLD_ds.question.values, "evidence": HOLD_ds.evidence.values,
                        "label": HOLD_ds.label.astype(int).values,
                        "meta": [{"qid": q, "bench": BENCH[q], "source": "kora-holdout-oracle", "site": s, "ts": float(t), "cand": str(c or "")}
                                 for q, s, t, c in zip(HOLD_ds.qid, HOLD_ds.site, HOLD_ds.ts, HOLD_ds.cand)]})
HOLD_ds.to_parquet(f"{V}/holdout.parquet", index=False)
print(f"texts with both labels dropped: {int(both.sum())}; duplicates removed: {len(keep[~both]) - len(HOLD_ds)}")
hb = HOLD_ds.assign(bench=HOLD_ds.meta.map(lambda m: m["bench"]))
t = hb.groupby("bench").agg(states=("label", "size"), questions=("meta", lambda m: len({x["qid"] for x in m})), yes=("label", "mean"))
t.loc["total"] = [len(hb), hb.meta.map(lambda m: m["qid"]).nunique(), hb.label.mean()]
t.round(3)

In [ ]:
# The holdout's pages (4.4), from its run's traces; then the checks of 4.4.
TRACES["holdout"] = [f"{RESULTS}/sj2-hold-oracle/qwen3-30b-a3b/{s}-train/kora_oracle_nomem/traces" for s in SETS]
PAGED["holdout"] = with_pages(f"{V}/holdout.parquet", logged_calls(TRACES["holdout"]))
ds = pd.read_parquet(f"{V}/holdout.parquet")
full = np.array([len(x) for x in tok(ds.question.tolist(), (ds.evidence + "\n\n" + ds.page).tolist())["input_ids"]])
rnd = ds.meta.map(lambda m: m["page_round"])
pd.Series({"rows": len(ds), **{f"page from round {r}": int((rnd == r).sum()) for r in ("1", "2", "3", "root")},
           "no page found (must be ~0)": int((rnd == "none").sum()),
           "LLM-directed text left (must be 0)": int(sum(any(w in p for w in LLM_TEXT) for p in ds.page)),
           "rows trimmed": int((ds.meta.map(lambda m: m["page_dropped"]) > 0).sum()),
           "rows over 7,999 after (must be 0)": int((full > LIMIT).sum())})

**Holdout states result:** 11,153 logged states on 956 questions (944 of 1,000 ran cleanly: 44 topic entities not in the dump, 12 timeouts); 2,357 sub-question states dropped; after labelling, validation and cleaning, 7,860 states, 23.7% yes, 165 to 200 questions per set; every state has its page.

## 5. Feature engineering

Following the ModernBERT / Ettin fine-tuning setup (ModernBERT's GLUE code, reused by Ettin): **no hand-made
features**. The encoder reads the text and learns its own features. What is decided here is only what text goes in
and how it is tokenised:

| setting | baseline | why |
|---|---|---|
| input | **text pair** `[CLS] question [SEP] evidence [SEP]` | the template of their own starter script for custom data (QNLI); `[SEP]` marks where the question ends |
| relation names | **raw**, as the graph writes them (`film.film.directed_by`, `DIRECTED_BY`) | as TIARA and SR; no paper compared raw against readable |
| candidate names | **30 shown**, then `(+N more)`; the count `(N items)` is always the true one | short inputs; the judge knows the full size |
| which graph | **not given** | a base-rate shortcut, and no name exists for an unseen graph; the relation names already show it |
| KORA's type restriction | **not shown** | KORA-specific (ToG, PoG have none) |
| tokenizer | **the model's own** (ModernBERT and Ettin share one) | token ids only mean something to their own model |
| length | **max 7,999 tokens, nothing truncated**; padding per batch, batches grouped by length | every row fits (longest 7,010); short rows cost the same |
| format | JSON not used | the same information in more tokens |

Later ablations, **not in the baseline**, one at a time if wanted: single block; readable relation names; full
candidate lists; an extra line (candidate kind, the graph's own count, the expected answer count); the graph name.

The **page** (§4.4, the schema the query agent saw) is stored as its own column and is not in the baseline; §6.1
tests adding it.


In [ ]:
# One input as the encoder receives it: a KORA state (B) and a generated row (A).
for name in ("B", "A"):
    d = pd.read_parquet(f"{V}/rendered/{name}.parquet")
    r = d[d.label == 1].sample(1, random_state=11).iloc[0]
    enc = tok(r.question, r.evidence, truncation=True, max_length=7999)
    ids = enc["input_ids"]; pieces = tok.convert_ids_to_tokens(ids)
    print(f"================ {name} ({'KORA state' if name == 'B' else 'generated'}) | label {r.label} | {r.meta['bench']}")
    print("what the model reads (decoded back from its token ids):\n")
    print(tok.decode(ids))
    print(f"\n{len(ids)} tokens; inputs passed to the model: {list(enc.keys())} (+ the label)")
    print("first 14 tokens:", pieces[:14])
    print("[SEP] at positions:", [k for k, t in enumerate(pieces) if t == "[SEP]"], "\n")

In [ ]:
# Rows per length bucket (results are reported per bucket).
edges, labels = [0, 128, 512, 2048, 7999], ["<=128", "129-512", "513-2,048", "2,049-7,999"]
out = {}
for name in ("A", "B"):
    d = pd.read_parquet(f"{V}/rendered/{name}.parquet")
    lens = pd.Series([len(x) for x in tok(d.question.tolist(), d.evidence.tolist())["input_ids"]])
    out[name] = pd.cut(lens, edges, labels=labels).value_counts().reindex(labels)
pd.DataFrame(out)

## 6. Model selection and evaluation
The procedure, fixed before any result:
1. **Screen** (successive halving: Jamieson & Talwalkar, AISTATS 2016; Hyperband, Li et al., JMLR 2018): logistic
   regression (grouped cross-validation on train), Ettin encoders 17M / 68M / 150M / 400M and ModernBERT-large, each
   on the same fixed 10% and 20% of the training rows, scored on the validation states; only the ranking is used.
2. **Finalists**: the top two plus ModernBERT-large on all training rows, 3 seeds, threshold on validation, one live
   validation run each. The best is the **base judge**, for A and for B separately.
3. **DAgger** from the base judge: model and recipe frozen, only the data grows; a round gains when its paired 95%
   interval against the previous round excludes zero; stop after two rounds without a gain.
4. **Holdout** once per experiment; **test** once at the end, on the question sets the baselines ran on, scores only.

Measured: disagreement with the oracle in both directions (offline, on labelled validation states; live, on the
judge's own states), Hit@1 and F1. Memory off throughout.

### 6.1 Does adding the subgraph schema help?
**Question**: does giving the judge the schema of the **subgraph the query
agent saw** lower its disagreement with the oracle? Two kinds of error might be visible in the schema but not in the
evidence: the **wrong kind of answer** (the last relation leads to an airport, the question wants an airline) and the
**wrong relation** (a better relation existed at the anchor and was not taken), including partial answers that stop a
step short.

| variant | the model reads | when |
|---|---|---|
| **0. baseline** | `question`, `evidence` (§5) | now |
| **1. subgraph seen** | `question`, `evidence` + `page` (§4.4: the page of the round the state came from, LLM text stripped, trimmed to fit) | now |
| 2. path used | `question`, `evidence` + only the page lines of the path's own relations | only if 1 helps, to show where the gain comes from |

**Setup**, the same for both variants:

| setting | value | from |
|---|---|---|
| training rows | B, the states of a fixed **20% of its questions** (all states of a question together); A+B after | our choice |
| validation | the 5,625 validation states (§4.3), memory off | §4.3 |
| model | Ettin 17M (`ettin-encoder-17m`), 2 classes, cross-entropy; the classifier head as the checkpoint configures it (mean pooling) | our choice; the checkpoint |
| optimiser | Composer's DecoupledAdamW on all parameters: betas (0.9, 0.98), eps 1e-6, weight decay 8e-6 applied as weights × (1 − 8e-6 × lr now / lr at start) each step | bert24 `src/evals/glue_jobs.py`; Composer `decoupled_weight_decay.py` |
| schedule | 6% warm-up, then linear decay to 0 | bert24 `yamls/finetuning/glue/*.yaml` (`t_warmup: 0.06dur`, `alpha_f: 0.0`) |
| batch | 32 rows (as 4 steps of 8, gradients summed, as their automatic micro-batches), padded per batch | Ettin App. F (batch {16, 32}) |
| batch order | rows of similar length together (theirs: plain shuffle; ours: 7,999-token rows) | our choice |
| precision | bf16 mixed | the GLUE yaml (`amp_bf16`) |
| gradient clipping | none | the GLUE yaml has no `algorithms` |
| grid | learning rate {3e-5, 8e-5} × epochs {1, 2}, for each variant: 8 fine-tunings. A subset of Ettin's GLUE grid (lr {1e-5, 3e-5, 5e-5, 8e-5, 1e-4} × weight decay {1e-6, 5e-6, 8e-6, 1e-5} × batch {16, 32} × epochs {1, 2, 3, 4} for the large tasks) | our choice; Ettin App. F |
| seed | 0 | |

The training loop is plain PyTorch (`accelerate`, which Hugging Face's `Trainer` needs, is not installed), written out
below so every step is visible; it reproduces their Composer recipe setting by setting (table). §6 runs on one H100 GPU.

**Scores** on validation, for every setting: the threshold that balances the FN and FP rates (chosen on validation),
then FN rate, FP rate, precision, recall, F1 and accuracy of "yes", and AUROC. Per set, per source of the candidate
(seed written by code, LLM query, round-3 search, root) and per page round. **The best setting of each variant** is the
one with the lower worse-of-FN-and-FP rate, F1 breaking ties.

**Decision rule:** variant 1 beats 0 when the **paired 95% interval** of its gain excludes zero: both best settings
scored on the same states, the validation questions resampled 1,000 times with all their states, the difference
recomputed each time, the interval the middle 95%. If 1 wins, the page is the default input from here on; if not, one
check on a larger model (150M) before dropping it.

In [ ]:
# a. The data: B's training rows on a fixed 20% of its questions, and the validation states.
import time, torch
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import roc_auc_score

# §6 starts from the saved datasets, so it can run in its own session (on the GPU node) after the imports of §1.
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(f"{DATA_DIR}/kora/models/ModernBERT-large")   # Ettin's, the same
LIMIT = 7999

TRAIN = pd.read_parquet(f"{V}/rendered/B.parquet")
VAL = pd.read_parquet(f"{V}/validation.parquet")
questions = sorted(TRAIN.meta.map(lambda m: m["qid"]).unique())
screen = set(random.Random(0).sample(questions, round(0.2 * len(questions))))
TRAIN20 = TRAIN[TRAIN.meta.map(lambda m: m["qid"] in screen)].reset_index(drop=True)
json.dump(sorted(screen), open(f"{OUT}/B_screen20.json", "w"))
pd.DataFrame({"rows": [len(TRAIN20), len(VAL)], "questions": [len(screen), VAL.meta.map(lambda m: m["qid"]).nunique()],
              "yes": [TRAIN20.label.mean().round(3), VAL.label.mean().round(3)]}, index=["train (20% of B)", "validation"])

In [ ]:
# The final datasets as saved, just before training: the CSV (the Parquet holds the same), five rows, and the counts.
for name, path in (("train: B", f"{V}/rendered/B.csv"), ("validation", f"{V}/validation.csv")):
    d = pd.read_csv(path)
    print(f"===== {name}: {path}\ncolumns: {list(d.columns)}  |  {len(d):,} rows")
    with pd.option_context("display.max_colwidth", 90):
        display(d.head(5))

source = lambda m: ("root" if m["site"] == "root" else "seed (code)" if re.fullmatch(r"s\d*", m["cand"] or "")
                    else "LLM query" if re.fullmatch(r"q\d+", m["cand"] or "") else "round-3 search")
def counts(d):
    m = pd.DataFrame(list(d.meta))
    plain = np.array([len(x) for x in tok(d.question.tolist(), d.evidence.tolist())["input_ids"]])
    paged = np.array([len(x) for x in tok(d.question.tolist(), (d.evidence + "\n\n" + d.page).tolist())["input_ids"]])
    out = {"rows": len(d), "questions": m.qid.nunique(), "yes": round(d.label.mean(), 3)}
    out.update({f"set {k}: rows": v for k, v in m.bench.value_counts().sort_index().items()})
    out.update({f"site {k}": v for k, v in m.site.value_counts().items()})
    out.update({f"source {k}": v for k, v in m.apply(source, axis=1).value_counts().items()})
    out.update({f"page round {k}": v for k, v in m.page_round.value_counts().sort_index().items()})
    out.update({"tokens without page: median": int(np.median(plain)), "  p99": int(np.percentile(plain, 99)),
                "tokens with page: median": int(np.median(paged)), "  p99 ": int(np.percentile(paged, 99)),
                "rows with the page trimmed": int((m.page_dropped > 0).sum())})
    return out
pd.DataFrame({"B, all": counts(TRAIN), "B, 20% (trained on here)": counts(TRAIN20), "validation": counts(VAL)}).fillna(0)

In [ ]:
# b. The model input, the batches and the fine-tuning: the ModernBERT / Ettin GLUE recipe (bert24 src/evals/glue_jobs.py,
# yamls/finetuning/glue/*.yaml; Ettin paper App. F).
ENCODER = f"{DATA_DIR}/kora/models/ettin-encoder-17m"
RECIPE = {"batch": 32, "warmup": 0.06, "weight_decay": 8e-6, "betas": (0.9, 0.98), "eps": 1e-6}

def inputs(df, with_page):
    """Token ids of the text pair; with the page, it follows the evidence in the second text."""
    second = (df.evidence + "\n\n" + df.page) if with_page else df.evidence
    ids = tok(df.question.tolist(), second.tolist())["input_ids"]
    assert max(map(len, ids)) <= LIMIT
    return ids

def batches(ids, labels, size, seed=None):
    """Rows of similar length together (sorted inside chunks of 50 batches), padded per batch; shuffled when seed is set."""
    order = np.random.default_rng(seed).permutation(len(ids)) if seed is not None else np.arange(len(ids))
    chunk = size * 50
    order = np.concatenate([sorted(order[k:k + chunk], key=lambda i: len(ids[i])) for k in range(0, len(order), chunk)])
    groups = [order[k:k + size] for k in range(0, len(order), size)]
    if seed is not None:
        np.random.default_rng(seed + 1).shuffle(groups)
    for g in groups:
        x = torch.full((len(g), max(len(ids[i]) for i in g)), tok.pad_token_id)
        mask = torch.zeros_like(x)
        for row, i in enumerate(g):
            x[row, :len(ids[i])] = torch.tensor(ids[i]); mask[row, :len(ids[i])] = 1
        yield g, x, mask, torch.tensor([labels[i] for i in g])

def finetune(ids, labels, lr, epochs, seed=0, micro_tokens=65536, encoder=ENCODER, progress=None, **config):   # config: e.g. classifier_pooling (§6.1b)
    torch.manual_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(encoder, num_labels=2, **config).cuda()
    # Composer's DecoupledAdamW on all parameters: each step the weights shrink by weight_decay x (lr now / lr at start),
    # not by lr x weight_decay. PyTorch's AdamW shrinks them by lr now x its weight_decay, so weight_decay / lr is the same.
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=RECIPE["betas"], eps=RECIPE["eps"],
                            weight_decay=RECIPE["weight_decay"] / lr)
    steps = epochs * -(-len(ids) // RECIPE["batch"])
    schedule = get_linear_schedule_with_warmup(opt, round(RECIPE["warmup"] * steps), steps)
    model.train()
    for epoch in range(epochs):
        for g, x, mask, y in batches(ids, labels, RECIPE["batch"], seed=seed + epoch):
            micro = max(1, min(len(g), micro_tokens // x.shape[1]))         # rows per step: all 32 when they fit in micro_tokens
            for k in range(0, len(g), micro):                                 # (memory only: gradients summed over the 32)
                part = slice(k, k + micro)
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    loss = model(input_ids=x[part].cuda(), attention_mask=mask[part].cuda(), labels=y[part].cuda()).loss
                (loss * len(y[part]) / len(g)).backward()
            opt.step(); schedule.step(); opt.zero_grad()
            done = schedule.last_epoch
            if progress and done % 100 == 0:                                    # a line every 100 steps, for long runs
                with open(progress, "a") as fh:
                    fh.write(f"{time.strftime('%H:%M:%S')} step {done}/{steps} loss {loss.item():.4f}\n")
    return model

@torch.no_grad()
def p_yes(model, ids, batch=32):
    """The model's probability of "yes" for every row."""
    model.eval(); out = np.zeros(len(ids))
    for g, x, mask, _ in batches(ids, [0] * len(ids), batch):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(input_ids=x.cuda(), attention_mask=mask.cuda()).logits.float()
        out[g] = torch.softmax(logits, -1)[:, 1].cpu().numpy()
    return out

In [ ]:
# c. The grid: 2 variants x learning rate {3e-5, 8e-5} x epochs {1, 2}. Each fine-tuning's validation scores are saved
# (a finished one is read back, not redone).
RUNS = f"{V}/models/schema_screen"; os.makedirs(RUNS, exist_ok=True)
SCORES = {}
for variant, with_page in (("baseline", False), ("page", True)):
    train_ids, val_ids = inputs(TRAIN20, with_page), inputs(VAL, with_page)
    for lr in (3e-5, 8e-5):
        for epochs in (1, 2):
            name = f"{variant}_lr{lr:g}_ep{epochs}"
            if os.path.exists(f"{RUNS}/{name}.npy"):
                SCORES[name] = np.load(f"{RUNS}/{name}.npy"); continue
            t0 = time.time()
            model = finetune(train_ids, TRAIN20.label.tolist(), lr, epochs)
            SCORES[name] = p_yes(model, val_ids); np.save(f"{RUNS}/{name}.npy", SCORES[name])
            print(f"{name}: {(time.time() - t0) / 60:.1f} min", flush=True)
            del model; torch.cuda.empty_cache()

In [ ]:
# d. Scores at the threshold that balances the FN and FP rates; the best setting of each variant.
y = VAL.label.to_numpy()

def balanced_threshold(y, s):
    best = (2.0, 0.5)
    for t in np.unique(np.quantile(s, np.linspace(0, 1, 401))):
        p = s >= t
        best = min(best, (max(((~p) & (y == 1)).sum() / max((y == 1).sum(), 1), (p & (y == 0)).sum() / max((y == 0).sum(), 1)), float(t)))
    return best[1]

def metrics(y, s, t):
    p = s >= t; tp = int((p & (y == 1)).sum()); fp = int((p & (y == 0)).sum()); fn = int(((~p) & (y == 1)).sum())
    return {"n": len(y), "FN rate": fn / max((y == 1).sum(), 1), "FP rate": fp / max((y == 0).sum(), 1),
            "precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1), "F1": 2 * tp / max(2 * tp + fp + fn, 1),
            "accuracy": float((p == (y == 1)).mean()), "AUROC": roc_auc_score(y, s) if 0 < y.sum() < len(y) else np.nan}

THRESH = {n: balanced_threshold(y, s) for n, s in SCORES.items()}
GRID = pd.DataFrame({n: {**metrics(y, s, THRESH[n]), "threshold": THRESH[n]} for n, s in SCORES.items()}).T
GRID["worse of FN, FP"] = GRID[["FN rate", "FP rate"]].max(axis=1)
BEST = {v: min((n for n in SCORES if n.startswith(v)), key=lambda n: (GRID.loc[n, "worse of FN, FP"], -GRID.loc[n, "F1"]))
        for v in ("baseline", "page")}
print("best setting per variant:", BEST)
GRID.drop(columns="n").round(3)

In [ ]:
# e. The best of each variant, by set, by the candidate's source and by the page's round.
seed_or_llm = lambda m: ("root" if m["site"] == "root" else "seed (code)" if re.fullmatch(r"s\d*", m["cand"] or "")
                         else "LLM query" if re.fullmatch(r"q\d+", m["cand"] or "") else "round-3 search")
GROUPS = {"set": VAL.meta.map(lambda m: m["bench"]), "source": VAL.meta.map(seed_or_llm), "page round": VAL.meta.map(lambda m: m["page_round"])}
for by, g in GROUPS.items():
    t = {(v, k): metrics(y[(g == k).to_numpy()], SCORES[BEST[v]][(g == k).to_numpy()], THRESH[BEST[v]])
         for v in BEST for k in sorted(g.unique())}
    display(pd.DataFrame(t).T[["n", "FN rate", "FP rate", "F1", "AUROC"]].unstack(0).round(3).rename_axis(by))

In [ ]:
# f. Paired bootstrap over validation questions: is the page's gain real? (thresholds fixed at the full-validation ones)
qid = VAL.meta.map(lambda m: m["qid"]).to_numpy()
rows_of = [np.flatnonzero(qid == q) for q in np.unique(qid)]
base, page = SCORES[BEST["baseline"]], SCORES[BEST["page"]]
diff = {"worse of FN, FP (lower is better)": [], "F1": [], "AUROC": []}
rng = np.random.default_rng(0)
for _ in range(1000):
    k = np.concatenate([rows_of[i] for i in rng.integers(0, len(rows_of), len(rows_of))])
    mb, mp = metrics(y[k], base[k], THRESH[BEST["baseline"]]), metrics(y[k], page[k], THRESH[BEST["page"]])
    diff["worse of FN, FP (lower is better)"].append(max(mp["FN rate"], mp["FP rate"]) - max(mb["FN rate"], mb["FP rate"]))
    diff["F1"].append(mp["F1"] - mb["F1"]); diff["AUROC"].append(mp["AUROC"] - mb["AUROC"])
pd.DataFrame({k: {"page − baseline": np.mean(v), "95% low": np.percentile(v, 2.5), "95% high": np.percentile(v, 97.5)}
              for k, v in diff.items()}).T.round(4)

**Result:** with mean pooling the page hurts at every setting. Best of each (lr 8e-5, 2 epochs): worse of FN/FP .373 (page) against .222 (baseline), AUROC .688 against .862, worse on every set; paired over questions: worse of FN/FP +.155 [+.121, +.194], F1 −.181, AUROC −.175. §6.1b finds the cause.

### 6.1b Diagnosis: why does the page hurt?
The page adds information and removes none, yet AUROC fell from .86 to .69. A model that found the page useless should
learn to ignore it, so the drop points to **how the page is fed**, not to what it says. Checked first (read-only):

- **Alignment is right:** every seed and LLM state's path relation is on its own page (100%, B and validation).
- **Train and validation pages differ:** relation lines per page, median **43 in B, 115 in validation**; CWQ pages
  9.6k against 21k characters, WebQSP 6.4k against 16.4k. Validation comes from a later KORA (arm E), whose pages hold
  more anchors and relations; 24% of validation rows are trimmed against 3.6% of B's.
- **The page is a question fingerprint:** one page is shared by about 5 states, 38% of B's pages carry only "no".

Hypotheses, each tested on Ettin 17M with the §6.1 recipe at its best setting (lr 8e-5, 2 epochs), same 20% and
validation states:

| # | hypothesis | test |
|---|---|---|
| H1 | mean pooling drowns the candidate: the page is ~95% of the tokens, the same for every candidate of a round | `cls` pooling (the first token) for baseline and page |
| H2 | the long page lets the model memorise questions (fits train, fails validation) | train AUROC beside validation AUROC for every run |
| H3 | the length shift (pages 2–3× longer in validation) | a **short page**: per block the first 8 relations in KORA's order (closest to the goal) plus every line with a path relation, the rest counted |
| H4 | only the path's own schema carries signal | the **path page**: only the lines of the path's relations (variant 2) |

In [ ]:
# Variants of the page, built from the stored one: short (8 relations per block + the path's) and path-only.
REL = [re.compile(r"^\s+(\S+): \S+ -> "), re.compile(r"\s(\S+) \?mid \. \?mid (\S+) \?answer"),   # as in §4.4
       re.compile(r"^\s+\?x (\S+) "), re.compile(r"\s(\S+) \?x\b")]
def relations_in(line):
    for rx in REL:
        m = rx.search(line)
        if m:
            return list(m.groups())
    return []

def path_rels(evidence):
    """The relations of the state's own path (the evidence lines `subject | relation | object`, not the header)."""
    return {x.split(" | ")[1].strip() for x in evidence.splitlines()[1:] if x.count(" | ") >= 2}

def cut_page(page, evidence, k):
    """Keep the first k relation lines of every block (KORA's order) and every line with a path relation; k=0: path lines only."""
    if not isinstance(page, str) or not page:
        return ""
    lines = page.splitlines()
    head, body = lines[0], lines[1:]
    keep_rel = path_rels(evidence)
    out, block, kept, seen = [head], None, [], 0
    def close():
        if block is not None and (kept or k):
            gone = seen - len(kept)
            out.extend([block] + kept + ([f"  (+{gone} more not shown)"] if gone > 0 else []))
    for l in body:
        if not l.startswith(" "):
            close(); block, kept, seen = l, [], 0
        elif not l.lstrip().startswith("(+"):
            seen += 1
            if seen <= k or set(relations_in(l)) & keep_rel:
                kept.append(l)
    close()
    return "\n".join(out) if len(out) > 1 else ""

for d in (TRAIN20, VAL):
    d["page_short"] = [cut_page(p, e, 8) for p, e in zip(d.page, d.evidence)]
    d["page_path"] = [cut_page(p, e, 0) for p, e in zip(d.page, d.evidence)]
L = lambda s: np.median([len(x) for x in tok(s.tolist(), add_special_tokens=False)["input_ids"]])
pd.DataFrame({c: {"train median tokens": L(TRAIN20[c]), "validation median tokens": L(VAL[c]),
                  "empty (train)": int((TRAIN20[c] == "").sum()), "empty (validation)": int((VAL[c] == "").sum())}
              for c in ("page", "page_short", "page_path")}).T

In [ ]:
# One example of each, for the same state.
r = VAL[(VAL.meta.map(lambda m: m["bench"]) == "cwq") & (VAL.page.str.len() > 3000)].iloc[0]
print(f"[question] {r.question}\n[evidence]\n{r.evidence}\n\n[page_short]\n{r.page_short}\n\n[page_path]\n{r.page_path}")

In [ ]:
# The runs: (input, pooling). Validation and train scores are saved; a finished run is read back.
def inputs_of(df, column):
    second = df.evidence if column is None else (df.evidence + "\n\n" + df[column]).str.rstrip()
    ids = tok(df.question.tolist(), second.tolist())["input_ids"]
    assert max(map(len, ids)) <= LIMIT
    return ids

DIAG = {"baseline_mean": (None, "mean"), "page_mean": ("page", "mean"),
        "baseline_cls": (None, "cls"), "page_cls": ("page", "cls"),
        "short_mean": ("page_short", "mean"), "short_cls": ("page_short", "cls"),
        "path_mean": ("page_path", "mean"), "path_cls": ("page_path", "cls")}
DIAG_SCORES = {}
for name, (column, pooling) in DIAG.items():
    f = f"{RUNS}/diag_{name}"
    if os.path.exists(f + "_val.npy"):
        DIAG_SCORES[name] = (np.load(f + "_val.npy"), np.load(f + "_train.npy")); continue
    t0 = time.time()
    tr_ids, va_ids = inputs_of(TRAIN20, column), inputs_of(VAL, column)
    model = finetune(tr_ids, TRAIN20.label.tolist(), 8e-5, 2, classifier_pooling=pooling)
    DIAG_SCORES[name] = (p_yes(model, va_ids), p_yes(model, tr_ids))
    np.save(f + "_val.npy", DIAG_SCORES[name][0]); np.save(f + "_train.npy", DIAG_SCORES[name][1])
    print(f"{name}: {(time.time() - t0) / 60:.1f} min", flush=True)
    del model; torch.cuda.empty_cache()

In [ ]:
# Validation (at its balanced threshold) and train AUROC: a large gap is memorisation.
y_tr = TRAIN20.label.to_numpy()
rows = {}
for name, (sv, st) in DIAG_SCORES.items():
    m = metrics(y, sv, balanced_threshold(y, sv))
    rows[name] = {"worse of FN, FP": max(m["FN rate"], m["FP rate"]), "FN rate": m["FN rate"], "FP rate": m["FP rate"],
                  "F1": m["F1"], "AUROC validation": m["AUROC"], "AUROC train": roc_auc_score(y_tr, st)}
DIAG_TABLE = pd.DataFrame(rows).T
DIAG_TABLE["train − validation AUROC"] = DIAG_TABLE["AUROC train"] - DIAG_TABLE["AUROC validation"]
DIAG_TABLE.round(3)

In [ ]:
# Paired bootstrap of every page variant against the baseline with the same pooling (thresholds fixed, 1,000 resamples
# of validation questions with all their states).
def paired(sb, sp, n=1000, seed=0):
    tb, tp = balanced_threshold(y, sb), balanced_threshold(y, sp)
    rng, out = np.random.default_rng(seed), {"worse of FN, FP": [], "F1": [], "AUROC": []}
    for _ in range(n):
        k = np.concatenate([rows_of[i] for i in rng.integers(0, len(rows_of), len(rows_of))])
        mb, mp = metrics(y[k], sb[k], tb), metrics(y[k], sp[k], tp)
        out["worse of FN, FP"].append(max(mp["FN rate"], mp["FP rate"]) - max(mb["FN rate"], mb["FP rate"]))
        out["F1"].append(mp["F1"] - mb["F1"]); out["AUROC"].append(mp["AUROC"] - mb["AUROC"])
    return {f"{k} {s}": f(v) for k, v in out.items()
            for s, f in (("change", np.mean), ("95% low", lambda v: np.percentile(v, 2.5)), ("95% high", lambda v: np.percentile(v, 97.5)))}

qid = VAL.meta.map(lambda m: m["qid"]).to_numpy()
rows_of = [np.flatnonzero(qid == q) for q in np.unique(qid)]
pd.DataFrame({f"{v}_{p} vs baseline_{p}": paired(DIAG_SCORES[f"baseline_{p}"][0], DIAG_SCORES[f"{v}_{p}"][0])
              for v in ("page", "short", "path") for p in ("mean", "cls")}).T.round(3)

**Diagnosis result:** the loss is the pooling, not the page. With mean pooling the page cannot even fit its training rows (train AUROC .730): the candidate is averaged away under ~2,000 page tokens. With `cls` pooling page and baseline are level (worse of FN/FP +.006 [−.016, +.029]; AUROC −.013 [−.023, −.002]), and path-only is level with the baseline under either pooling (cls +.001 [−.020, +.024]). `cls` pooling is used for every judge.

### 6.1c The bar, the headroom, all of B, a marked page, seeds
After 6.1b: with `cls` pooling the page no longer hurts, but does not clearly help on 20% of B. Five checks, all with `cls` pooling, lr 8e-5, 2 epochs:

| # | check | why |
|---|---|---|
| 1 | **r4** (the DAgger round-4 judge, ModernBERT-large, trained on 159k old-format rows of 8,221 train questions) scored on these validation states, exactly as its live server scores (the joined text, 1,024 tokens, sigmoid) | the bar. None of the 666 validation questions are in r4's training data, but its checkpoint and threshold were chosen on 664 of them (old-format states), so it is somewhat optimistic |
| 2 | **headroom**: the baseline's false accepts split into "no overlap with the gold" (a wrong relation or kind: what a schema can expose) and "partial" (some gold, below the .5 rule: a schema cannot tell) | how much any page could gain |
| 3 | **all of B** (37,155 rows): baseline, page, short page, path page | whether more data lets the model use the page |
| 4 | **marked page**: the page with the lines of the candidate's own relations marked `>>` (20% of B) | whether pointing the model at its own lines helps |
| 5 | **seeds 1 and 2** for baseline, page and path (20% of B) | which differences are larger than seed noise |

In [ ]:
# 1. r4 on the validation states, as its server scores them: the joined text (render.py, the server's own text),
# 1,024 tokens, one sigmoid output.
import sys
from estimator.render import joined
from transformers import AutoTokenizer as _AT
R4 = f"{DATA_DIR}/stopjudge/judges/dag_r4"
if os.path.exists(f"{RUNS}/r4_val.npy"):
    R4_SCORES = np.load(f"{RUNS}/r4_val.npy")
else:
    r4_tok = _AT.from_pretrained(f"{R4}/ckpt")
    r4 = AutoModelForSequenceClassification.from_pretrained(f"{R4}/ckpt").cuda().eval()
    texts = [joined(q, e) for q, e in zip(VAL.question, VAL.evidence)]
    out = []
    with torch.no_grad():
        for k in range(0, len(texts), 32):
            enc = r4_tok(texts[k:k + 32], truncation=True, max_length=1024, padding=True, return_tensors="pt").to("cuda")
            with torch.autocast("cuda", dtype=torch.bfloat16):
                out += torch.sigmoid(r4(**enc).logits.squeeze(-1).float()).cpu().tolist()
    R4_SCORES = np.array(out); np.save(f"{RUNS}/r4_val.npy", R4_SCORES)
    del r4; torch.cuda.empty_cache()
R4_THRESHOLD = json.load(open(f"{R4}/judge.json"))["threshold"]
pd.DataFrame({"r4 at its own threshold (live)": metrics(y, R4_SCORES, R4_THRESHOLD),
              "r4 at the balanced threshold": metrics(y, R4_SCORES, balanced_threshold(y, R4_SCORES)),
              "Ettin 17M baseline, cls, 20% of B": metrics(y, DIAG_SCORES["baseline_cls"][0], balanced_threshold(y, DIAG_SCORES["baseline_cls"][0]))}).T.round(3)

In [ ]:
# 2. Headroom: the baseline's (cls, 20%) errors at its balanced threshold, against the gold of each state.
GOLD = {}
for s in SETS:
    qs = json.load(open(f"{RAW}/questions/{s}/val.json"))
    for q in (qs["questions"] if isinstance(qs, dict) else qs):
        lit = lambda v: v if isinstance(v, list) else (ast.literal_eval(v) if v else [])
        GOLD[str(q["id"])] = ({str(a) for a in lit(q.get("answer_ids"))}, {norm(a) for a in lit(q.get("answers"))})
logged = {(str(r["qid"]), round(r["ts"], 3), str(r.get("cand") or "")): r
          for f in glob.glob(f"{V}/runs/val_oracle2/harvest/*.jsonl") for r in map(json.loads, open(f))}
def overlap(m):
    r = logged.get((m["qid"], round(m["ts"], 3), m["cand"] or ""))
    if r is None:
        return np.nan
    ids, nms = [str(i) for i in r.get("ids") or []], [norm(n) for n in r.get("names") or []]
    gi, gn = GOLD.get(m["qid"], (set(), set()))
    return sum((i in gi) or (k < len(nms) and nms[k] in gn) for k, i in enumerate(ids))
VAL["gold_overlap"] = VAL.meta.map(overlap)
sb = DIAG_SCORES["baseline_cls"][0]; pred = sb >= balanced_threshold(y, sb)
fa, fr = pred & (y == 0), ~pred & (y == 1)
src = VAL.meta.map(lambda m: "root" if m["site"] == "root" else "seed" if re.fullmatch(r"s\d*", m["cand"] or "") else "LLM" if re.fullmatch(r"q\d+", m["cand"] or "") else "round 3")
print(f"states matched to their logged ids: {VAL.gold_overlap.notna().mean():.1%}")
pd.DataFrame({"false accepts (oracle no, judge yes)": {"all": int(fa.sum()),
                  "no overlap with the gold (wrong relation or kind)": int((fa & (VAL.gold_overlap == 0)).sum()),
                  "partial (some gold, below the .5 rule)": int((fa & (VAL.gold_overlap > 0)).sum())},
              "false rejects (oracle yes, judge no)": {"all": int(fr.sum()), "seed": int((fr & (src == "seed")).sum()),
                  "LLM": int((fr & (src == "LLM")).sum()), "root": int((fr & (src == "root")).sum())}})

In [ ]:
# 3-5. The runs (cls pooling, lr 8e-5, 2 epochs). Validation scores saved; a finished run is read back.
TRAIN["page_short"] = [cut_page(p, e, 8) for p, e in zip(TRAIN.page, TRAIN.evidence)]
TRAIN["page_path"] = [cut_page(p, e, 0) for p, e in zip(TRAIN.page, TRAIN.evidence)]
def marked(page, evidence):
    rels = path_rels(evidence)
    return "\n".join((">> " + l[2:]) if l.startswith("  ") and set(relations_in(l)) & rels else l for l in str(page or "").splitlines())
for d in (TRAIN20, VAL):
    d["page_marked"] = [marked(p, e) for p, e in zip(d.page, d.evidence)]
    n = np.array([len(x) for x in tok(d.question.tolist(), (d.evidence + "\n\n" + d.page_marked).tolist())["input_ids"]])
    d.loc[n > LIMIT, "page_marked"] = d.loc[n > LIMIT, "page"]           # marking can add a token or two
    print(f"marked page kept unmarked (over {LIMIT} tokens when marked): {int((n > LIMIT).sum())}")
MORE = {}
plan = [("all_baseline", TRAIN, None, 0), ("all_path", TRAIN, "page_path", 0), ("all_page", TRAIN, "page", 0),
        ("all_short", TRAIN, "page_short", 0), ("marked", TRAIN20, "page_marked", 0)]
plan += [(f"{v}_seed{s}", TRAIN20, c, s) for s in (1, 2) for v, c in (("baseline", None), ("path", "page_path"), ("page", "page"))]
for name, data, column, seed in plan:
    f = f"{RUNS}/more_{name}_val.npy"
    if os.path.exists(f):
        MORE[name] = np.load(f); continue
    t0 = time.time()
    model = finetune(inputs_of(data, column), data.label.tolist(), 8e-5, 2, seed=seed, classifier_pooling="cls")
    MORE[name] = p_yes(model, inputs_of(VAL, column)); np.save(f, MORE[name])
    print(f"{name}: {(time.time() - t0) / 60:.1f} min", flush=True)
    del model; torch.cuda.empty_cache()

In [ ]:
# Results: every run at its balanced threshold, r4 beside them; seeds as mean ± sd; then paired bootstraps.
def row(s):
    m = metrics(y, s, balanced_threshold(y, s))
    return {"worse of FN, FP": max(m["FN rate"], m["FP rate"]), "FN rate": m["FN rate"], "FP rate": m["FP rate"], "F1": m["F1"], "AUROC": m["AUROC"]}
table = {"r4 (balanced threshold)": row(R4_SCORES)}
for v in ("baseline", "path", "page"):
    runs = [DIAG_SCORES[f"{v}_cls"][0], MORE[f"{v}_seed1"], MORE[f"{v}_seed2"]]
    rs = pd.DataFrame([row(s) for s in runs])
    table[f"20% {v}, 3 seeds: mean"] = rs.mean().to_dict(); table[f"20% {v}, 3 seeds: sd"] = rs.std().to_dict()
table["20% marked page"] = row(MORE["marked"])
for v in ("baseline", "path", "page", "short"):
    table[f"all of B: {v}"] = row(MORE[f"all_{v}"])
display(pd.DataFrame(table).T.round(3))
pd.DataFrame({"all of B: path vs baseline": paired(MORE["all_baseline"], MORE["all_path"]),
              "all of B: page vs baseline": paired(MORE["all_baseline"], MORE["all_page"]),
              "all of B: short vs baseline": paired(MORE["all_baseline"], MORE["all_short"]),
              "20%: marked vs baseline (seed 0)": paired(DIAG_SCORES["baseline_cls"][0], MORE["marked"]),
              "all of B baseline vs r4": paired(R4_SCORES, MORE["all_baseline"])}).T.round(3)

**6.1c result:** (1) The bar: r4 at the balanced threshold, worse of FN/FP .087 and AUROC .964 on the validation states; the 17M on 20% of B is far below (.203 / .876). (2) Headroom: of the 17M's 925 false accepts, 691 (75%) share nothing with the gold (a wrong relation or kind) and 179 are partial; of its 227 false rejects, 111 are seeds, 69 LLM queries and 41 root states. (3–5) Three seeds of the 20% baseline give .199 ± .007; path-only (.207 ± .011), the page (.227 ± .026) and the marked page (.206) sit inside that spread. All of B: .130 (baseline), .125 (path), .136 (page), .129 (short); path against baseline −.005 [−.022, +.014]. The schema stays an ablation.

### 6.1d Ettin 400M on all of B: baseline and path-only
At 17M no schema variant beats the baseline (6.1c). Does a larger model use the path's schema? Ettin 400M (Large: 28
layers, hidden 1,024), all of B, `cls` pooling, the same recipe with **lr 3e-5** (the low-middle of Ettin's App. F grid;
the table of Ettin's model sizes lists *pre-training* learning rates, not fine-tuning ones), 2 epochs, one seed. Long rows
are split into micro-batches of at most 16k tokens (memory only; the batch of 32 and its summed gradient are unchanged).

In [ ]:
# Ettin 400M: baseline and path-only on all of B (cls, lr 3e-5, 2 epochs).
LARGE = f"{DATA_DIR}/kora/models/ettin-encoder-400m"
for name, column in (("large_all_baseline", None), ("large_all_path", "page_path")):
    f = f"{RUNS}/more_{name}_val.npy"
    if os.path.exists(f):
        MORE[name] = np.load(f); continue
    t0 = time.time()
    model = finetune(inputs_of(TRAIN, column), TRAIN.label.tolist(), 3e-5, 2, micro_tokens=16384, encoder=LARGE,
                     progress=f"{RUNS}/more_{name}.progress", classifier_pooling="cls")
    MORE[name] = p_yes(model, inputs_of(VAL, column), batch=8); np.save(f, MORE[name])
    print(f"{name}: {(time.time() - t0) / 60:.1f} min", flush=True)
    del model; torch.cuda.empty_cache()
display(pd.DataFrame({"r4 (balanced threshold)": row(R4_SCORES), "17M all of B: baseline": row(MORE["all_baseline"]),
                      "17M all of B: path": row(MORE["all_path"]), "400M all of B: baseline": row(MORE["large_all_baseline"]),
                      "400M all of B: path": row(MORE["large_all_path"])}).T.round(3))
pd.DataFrame({"400M: path vs baseline": paired(MORE["large_all_baseline"], MORE["large_all_path"]),
              "400M vs 17M (baseline)": paired(MORE["all_baseline"], MORE["large_all_baseline"]),
              "400M baseline vs r4": paired(R4_SCORES, MORE["large_all_baseline"])}).T.round(3)

**6.1d result:** Ettin 400M on all of B: worse of FN/FP .075 and AUROC .976 (baseline), .073 / .978 (path-only). 400M against 17M: −.056 [−.075, −.040] worse of FN/FP, +.104 F1, +.035 AUROC; against r4: −.013 [−.029, +.004] worse of FN/FP, +.026 [+.008, +.042] F1, +.012 [+.006, +.018] AUROC. Path-only against the baseline stays level (−.002 [−.017, +.010]). The 400M on the baseline input is the judge from here on.

## 6.2 Round-0 judges: Ettin 400M, saved
The 400M on all of B (§6.1d) is level with r4 on validation. Four round-0 judges are trained and **saved** (model,
tokenizer and `judge.json` with the validation threshold) under `models/round0/<name>/`, each with the §6.1d recipe
(`cls` pooling, lr 3e-5, 2 epochs, seed 0):

| judge | data | input |
|---|---|---|
| `B` | all of B (37,155) | question, evidence |
| `B_path` | all of B | question, evidence + the path's schema lines |
| `A` | all of A (84,735): generated from the graph and the gold, no agent run | question, evidence |
| `AB` | A and B: texts present in both kept once (B's row), texts with both labels dropped | question, evidence |

`B` goes live first: KORA arm E with memory off on the 666 validation questions, the judge deciding at the leaf and the
root, as r4 was run (Search-on-Graph's run on the same questions stands). `B_path` cannot go live yet: KORA would have to
send the round's page to the judge server.

In [ ]:
# Train, score on validation, save: the model and tokenizer, and judge.json (threshold balanced on validation).
ROUND0 = f"{V}/models/round0"

def train_and_save(name, data, column, note, root=None):
    out = f"{root or ROUND0}/{name}"
    if os.path.exists(f"{out}/judge.json"):
        return json.load(open(f"{out}/judge.json"))
    t0 = time.time()
    model = finetune(inputs_of(data, column), data.label.tolist(), 3e-5, 2, micro_tokens=16384, encoder=LARGE,
                     progress=f"{RUNS}/round0_{name}.progress", classifier_pooling="cls")
    s = p_yes(model, inputs_of(VAL, column), batch=8)
    t = balanced_threshold(y, s)
    os.makedirs(out, exist_ok=True)
    model.save_pretrained(f"{out}/ckpt"); tok.save_pretrained(f"{out}/ckpt"); np.save(f"{out}/validation_scores.npy", s)
    info = {"name": name, "threshold": float(t), "max_len": LIMIT, "encoder": LARGE, "input": column or "question, evidence",
            "data": note, "rows": len(data), "lr": 3e-5, "epochs": 2, "pooling": "cls", "seed": 0,
            "validation": {k: float(v) for k, v in metrics(y, s, t).items()}, "minutes": round((time.time() - t0) / 60, 1)}
    json.dump(info, open(f"{out}/judge.json", "w"), indent=1)
    del model; torch.cuda.empty_cache()
    return info

ROUND0_INFO = {"B": train_and_save("B", TRAIN, None, "all of B")}
pd.DataFrame({k: v["validation"] | {"threshold": v["threshold"], "minutes": v["minutes"]} for k, v in ROUND0_INFO.items()}).T.round(3)

In [ ]:
# A, and A with B: A's dataset as saved in §4.2; when mixed, a text in both is kept once (B's row) and a text carrying
# both labels is dropped (§4.2 found 10,234 shared and 39 conflicting).
A_TRAIN = pd.read_parquet(f"{V}/rendered/A.parquet")
mix = pd.concat([TRAIN[["question", "evidence", "label", "meta"]], A_TRAIN[["question", "evidence", "label", "meta"]]], ignore_index=True)
both = mix.groupby(["question", "evidence"]).label.transform("nunique") > 1
AB_TRAIN = mix[~both].drop_duplicates(["question", "evidence"], keep="first").reset_index(drop=True)   # B comes first
print(f"A {len(A_TRAIN):,} + B {len(TRAIN):,} = {len(mix):,}; texts with both labels: {int(both.sum())} rows dropped; "
      f"duplicates removed: {len(mix) - int(both.sum()) - len(AB_TRAIN):,}; A+B: {len(AB_TRAIN):,} rows, {AB_TRAIN.label.mean():.3f} yes")
ROUND0_INFO["B_path"] = train_and_save("B_path", TRAIN, "page_path", "all of B")
ROUND0_INFO["A"] = train_and_save("A", A_TRAIN, None, "all of A")
ROUND0_INFO["AB"] = train_and_save("AB", AB_TRAIN, None, "A and B, de-duplicated")
display(pd.DataFrame({k: v["validation"] | {"threshold": v["threshold"], "rows": v["rows"], "minutes": v["minutes"]}
                      for k, v in ROUND0_INFO.items()}).T.round(3))
pd.DataFrame({f"{k} vs B": paired(np.load(f"{ROUND0}/B/validation_scores.npy"), np.load(f"{ROUND0}/{k}/validation_scores.npy"))
              for k in ("B_path", "A", "AB")} | {"B vs r4": paired(R4_SCORES, np.load(f"{ROUND0}/B/validation_scores.npy"))}).T.round(3)

**6.2 result:** four round-0 judges, weights saved. On the validation states: B worse of FN/FP .077, AUROC .975; B_path .076 / .977; A (generated data alone) .099 / .960, worse than B (F1 −.042 [−.060, −.024]); A+B (112,589 rows) .073 / .976, level with B (F1 +.009 [−.007, +.025]). Live, KORA memory off, the 666 validation questions: A+B Hit .800 / F1 .694, against the memory-off gold stop .839 / .709 (§8). A+B is π̂₁ of §7 and, after §7, the final judge (§9).

## 6.3 The holdout, read once
Every saved round-0 judge (6.2) and the two old judges (r0: ModernBERT-large on Dataset D; r4: after four DAgger rounds)
on the holdout states (4.4b), **at the threshold each chose on validation** (balanced FN/FP there): nothing is chosen
on the holdout. The old judges read the joined text at 1,024 tokens, as their server does.

In [ ]:
# Score every judge on the holdout states (scores saved beside each judge), at its validation threshold.
HOLDS = pd.read_parquet(f"{V}/holdout.parquet"); yh = HOLDS.label.to_numpy()
HOLDS["page_path"] = [cut_page(p, e, 0) for p, e in zip(HOLDS.page, HOLDS.evidence)]

def holdout_new(name):
    d = f"{ROUND0}/{name}"; info = json.load(open(f"{d}/judge.json")); f = f"{d}/holdout_scores.npy"
    if not os.path.exists(f):
        model = AutoModelForSequenceClassification.from_pretrained(f"{d}/ckpt").cuda()
        column = None if info["input"] == "question, evidence" else info["input"]
        np.save(f, p_yes(model, inputs_of(HOLDS, column), batch=8)); del model; torch.cuda.empty_cache()
    return np.load(f), info["threshold"]

def holdout_old(name, ckpt, val_scores):
    f = f"{RUNS}/{name}_holdout.npy"
    if not os.path.exists(f):
        old_tok = _AT.from_pretrained(ckpt); old = AutoModelForSequenceClassification.from_pretrained(ckpt).cuda().eval()
        texts, out = [joined(q, e) for q, e in zip(HOLDS.question, HOLDS.evidence)], []
        with torch.no_grad():
            for k in range(0, len(texts), 32):
                enc = old_tok(texts[k:k + 32], truncation=True, max_length=1024, padding=True, return_tensors="pt").to("cuda")
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    out += torch.sigmoid(old(**enc).logits.squeeze(-1).float()).cpu().tolist()
        np.save(f, np.array(out)); del old; torch.cuda.empty_cache()
    return np.load(f), balanced_threshold(y, val_scores)

HOLDOUT = {"r0 (old)": holdout_old("r0", f"{DATA_DIR}/stopjudge/judge_D_modernbert/ckpt", np.load(f"{RUNS}/r0_val.npy")),
           "r4 (old)": holdout_old("r4", f"{R4}/ckpt", R4_SCORES)}
for name in ("B", "B_path", "A", "AB"):
    HOLDOUT[name] = holdout_new(name)
def at(s, t):
    m = metrics(yh, s, t)
    return {"worse of FN, FP": max(m["FN rate"], m["FP rate"]), "FN rate": m["FN rate"], "FP rate": m["FP rate"], "F1": m["F1"], "AUROC": m["AUROC"]}
print(f"holdout: {len(HOLDS):,} states, {HOLDS.meta.map(lambda m: m['qid']).nunique()} questions, {yh.mean():.3f} yes")
pd.DataFrame({k: at(s, t) for k, (s, t) in HOLDOUT.items()}).T.round(3)

In [ ]:
# Paired bootstrap on the holdout (questions resampled; each judge at its validation threshold).
hq = HOLDS.meta.map(lambda m: m["qid"]).to_numpy()
hrows = [np.flatnonzero(hq == q) for q in np.unique(hq)]
def paired_holdout(a, b, n=1000):
    (sa, ta), (sb, tb) = HOLDOUT[a], HOLDOUT[b]; rng = np.random.default_rng(0); out = {"worse of FN, FP": [], "F1": [], "AUROC": []}
    for _ in range(n):
        k = np.concatenate([hrows[i] for i in rng.integers(0, len(hrows), len(hrows))])
        ma, mb = metrics(yh[k], sa[k], ta), metrics(yh[k], sb[k], tb)
        out["worse of FN, FP"].append(max(mb["FN rate"], mb["FP rate"]) - max(ma["FN rate"], ma["FP rate"]))
        out["F1"].append(mb["F1"] - ma["F1"]); out["AUROC"].append(mb["AUROC"] - ma["AUROC"])
    return {f"{k} {s}": f(v) for k, v in out.items()
            for s, f in (("change", np.mean), ("95% low", lambda v: np.percentile(v, 2.5)), ("95% high", lambda v: np.percentile(v, 97.5)))}
pd.DataFrame({f"{b} vs {a}": paired_holdout(a, b) for a, b in (("r4 (old)", "B"), ("r4 (old)", "AB"), ("r4 (old)", "A"),
                                                             ("B", "B_path"), ("B", "AB"), ("B", "A"))}).T.round(3)

**Holdout result (read once):** 7,860 states on 956 questions. A+B: worse of FN/FP .081, F1 .853, AUROC .979; against r4 −.016 [−.023, −.009] worse of FN/FP, +.025 [+.014, +.036] F1, +.013 [+.009, +.017] AUROC; against B level (−.005 [−.011, +.002]). A alone is worse than B (+.018 [+.010, +.026]).

## 6.4 KQA Pro as an unseen dataset
Does the judge work on a dataset and graph it never trained on, when the agent's state is still a path? KQA Pro (its
own graph, 388k triples) is held out from the judge: A's KQA Pro rows (10,681 rows, 793 questions) are removed and the
A+B judge is retrained without them (`AB_noKQA`, same recipe, saved). KQA Pro also stays out of every DAgger round.
Evaluation set: the 1,200 KQA Pro questions already in our format (1,000 from the training file, used only for A; 200
from validation, the holdout's KQA Pro part); the test sets are never touched. KORA runs on
them with memory off: once with the oracle (states, labels, ceiling) and once with `AB_noKQA` deciding.

In [ ]:
# A judge that has never seen KQA Pro: A+B without A's KQA Pro rows, the same recipe, saved as round0/AB_noKQA.
AB_NOKQA = AB_TRAIN[AB_TRAIN.meta.map(lambda m: m["bench"]) != "kqapro"].reset_index(drop=True)
print(f"A+B {len(AB_TRAIN):,} rows -> without KQA Pro {len(AB_NOKQA):,} ({len(AB_TRAIN) - len(AB_NOKQA):,} KQA Pro rows removed)")
ROUND0_INFO["AB_noKQA"] = train_and_save("AB_noKQA", AB_NOKQA, None, "A and B without KQA Pro")
pd.DataFrame({k: ROUND0_INFO[k]["validation"] | {"rows": ROUND0_INFO[k]["rows"]} for k in ("AB", "AB_noKQA")}).T.round(3)

**KQA Pro result:** the judge without KQA Pro (AB_noKQA, 101,909 rows) is level with A+B on the five-set validation (worse of FN/FP .078 against .073, AUROC .972 against .976). Its first live KQA Pro run is void: the judge server did not name KQA Pro ids, so the judge read `KQAPro:3401` where training showed the name (fixed in §7.8). The KQA Pro results are in §7.8.

## 7. DAgger rounds
**The idea in one line:** run KORA with the current judge, label every state it reaches with the gold, add those rows to
everything we have, train a new judge, and keep going while the live validation score improves: rounds of data
collection, stopped the way training is stopped on a validation loss.

**The method:** DAgger (Ross, Gordon & Bagnell, AISTATS 2011, Algorithm 3.1, read in full). Start from the expert's
data; each round run the current policy, label the states it visits with the expert, aggregate, train the next policy
on the union; return the best policy on validation. β = I(i = 1), "the simple, parameter-free version ... which often
performs best in practice": the expert drives only the first round. Our expert is the oracle rule on the gold (recall ≥ .5
and precision ≥ .5, by ids and names); round 0 is done (B is KORA driven by the oracle; the A+B judge is π̂₁, §6.2).

### 7.1 The loop, read like training with early stopping
| training a model | DAgger rounds |
|---|---|
| epoch | round |
| a pass over new batches | KORA run with the current judge on new training questions; every state it reaches, labelled |
| training set | A + B + every round so far (grows each round) |
| validation loss | the live validation run (666 questions, KORA memory off): **F1** first, Hit beside it, paired 95% intervals against the previous round; offline FN/FP on the fixed validation states and on the judge's own states as diagnostics |
| early stopping, patience 2 | a round is kept when its live F1 (or Hit) interval against the previous round excludes zero; **stop after 2 rounds without a gain**, at most 4 rounds on KORA |
| best checkpoint | the round with the best live validation score |
| test set, read once | the holdout, offline and live, for the best round only |

### 7.2 The data of one round
| | |
|---|---|
| questions | **1,000 fresh training questions: 200 per set** (CWQ, WebQSP, GrailQA, MetaQA, BioCQ), drawn with a fixed seed, never in any split, not test by id or by text (§2's checks), disjoint across rounds; GrailQA's drawn only where the topic entity is in our graph (35 holdout questions were lost to that). Fresh per set: CWQ 29,398, GrailQA ~42,900, WebQSP 1,725, BioCQ 3,384, **MetaQA 695 (3 rounds; a 4th reuses round 1's MetaQA)**. KQA Pro never (it is the unseen dataset, §6.4) |
| KORA run | arm E (`eval_overlay_dag`), **memory off**, the current judge at its validation threshold deciding at the leaf and the root, `KORA_HARVEST_AT_AGENT=1`, call traces on; run alone; ~25 min |
| what is kept | every **whole-question state** (the T0 leaf and the root), as B and validation; sub-question states dropped (no gold for them) |
| one row | `question`, `evidence` (`render.py`), `page` (the logged page, §4.4), `label` (the oracle rule on the gold), `meta` {qid, bench, source `round{i}`, site, ts, cand, page_round}: the format of every dataset so far |
| cleaning | as B: validation, texts with both labels dropped, duplicates removed within the round and against what is aggregated |
| saved | `splits/round{i}.json` (the questions), `round{i}/train.parquet` (+ `.csv`) |
| expected size | ~8,000 rows (validation gave 5,625 states from 666 questions), against A+B's 112,589 |

### 7.3 Train, gate, keep or stop
- **Train:** a fresh Ettin 400M on A + B + rounds 1..i (unweighted), the frozen recipe of §6.1d (`cls`, lr 3e-5, 2 epochs,
  batch 32), weights saved under `models/round{i}/` with `judge.json`; threshold balanced on the validation states.
- **Gate:** its server (parity check against the notebook first), the live validation run, scored with the authors' Hit
  rule, Hit@1, F1 and tokens; paired intervals against the previous round and against the memory-off oracle.
- **Keep / stop:** as 7.1. Every number goes into one table per round in this section.
- **Time:** rollout ~25 min, labelling ~10, training ~25, gate ~20: **about 1.5 h a round**, ~6 h for four.

### 7.4 What to expect
The old rounds took FN/FP from .108 to .087 over four rounds, with the live gain only at round 4. Our A+B judge is
already at the memory-off oracle on F1 (.694 against .697), so the rounds should mainly lower FN/FP on the judge's own
states (holdout .081 now) and close GrailQA (Hit .669 against the oracle's .787). On its own trajectory the best judged
candidate would give F1 .744: reaching that needs a different expert ("accept only the best candidate"), a separate
experiment after these rounds.

### 7.5 Rounds 5–7: other agents
The same loop on Think-on-Graph and Plan-on-Graph: a hook at each agent's stop decision turning its state (the paths and
the entities at their ends) into question / evidence / candidate; first the frozen judge zero-shot, then rounds on the
agent's own states. To settle: where the hook sits, what the candidate is when the agent
answers in free text, which questions, and the anonymised graphs.

### 7.6 Decisions
1. 1,000 questions a round, 200 per set.
2. Fresh questions each round (MetaQA allows 3).
3. The label stays the oracle rule (DAgger as written); a "best candidate" expert is a later, separate experiment.
4. The gate leads with live F1, Hit beside it.

### 7.7 The round cells
One round R (`SJ_ROUND=R` in the environment) runs these cells, each where its resources are; the KORA runs and the judge servers are started outside
the notebook:

| step | cells | where |
|---|---|---|
| draw the questions | §1 (`Q`, gold), §3.6 (graph access), 7a | CPU node |
| rollout | KORA arm E, memory off, the current judge (π̂_R: the A+B judge for R = 1, else round R−1's), harvest and traces on, run alone | CPU job |
| label | §1–4.4b (the helpers), 7b | CPU node |
| train | §6 (the recipe), 7c | H100 |
| gate | 7d (parity of the new judge's server), a KORA run on the 666 validation questions, 7e (score and decide) | H100 server, CPU job, CPU node |

Everything a round produces: `splits/round{R}.json`, `round{R}/train.parquet`, `models/dagger/round{R}/`
(weights, `judge.json`, validation scores), `runs/dagger/rounds.json` (one line per round: scores, intervals, the decision).

In [ ]:
# 7a. Round R's questions: 200 per set, fresh (in no split, no earlier round, not test by id or by text), drawn with a
# fixed seed; GrailQA's only where every topic entity has a name in our Freebase (35 holdout questions were lost to that).
import time
R = int(os.environ.get("SJ_ROUND", "1"))
DAGDIR = f"{V}/runs/dagger"; os.makedirs(DAGDIR, exist_ok=True)
SETS5, PER_SET = ["cwq", "webqsp", "grailqa", "metaqa", "biocq"], 200
taken = set()
for f in glob.glob(f"{OUT}/*.json"):
    if not f.endswith(f"/round{R}.json"):
        d = json.load(open(f))
        taken |= {str(q) for q in (d["questions"] if isinstance(d, dict) else d)}     # split records, or plain lists
TEST = json.load(open(f"{RAW}/test_questions.json"))
test_ids = {str(x["id"]) for v in TEST.values() for x in v}
test_txt = {norm(x["question"]) for v in TEST.values() for x in v if x["question"]}
def topic_ids(x):
    t = x.get("topic_entity") or {}
    return list((t if isinstance(t, dict) else ast.literal_eval(t)).keys())
def has_names(item):
    s, x = item
    ids = topic_ids(x)
    try:
        nm = names(s, ids)
        return str(x["id"]), bool(ids) and all(nm.get(i) for i in ids)
    except Exception:
        return str(x["id"]), False
drawn = {}
for s in SETS5:
    free = [x for x in Q[s] if str(x["id"]) not in taken and str(x["id"]) not in test_ids and norm(x["question"]) not in test_txt
            and (GOLD_IDS[str(x["id"])] or GOLD_NAMES[str(x["id"])]) and str(x.get("in_dump", "True")) != "False"]
    random.Random(f"round{R}-{s}").shuffle(free)
    if s == "grailqa":                                    # check a few more than needed, keep the first 200 that pass
        ok = dict(run_parallel(has_names, [(s, x) for x in free[:PER_SET * 2]], "grailqa topic names"))
        free = [x for x in free[:PER_SET * 2] if ok.get(str(x["id"]))]
    pick = [str(x["id"]) for x in free[:PER_SET]]
    if len(pick) < PER_SET:                               # MetaQA runs out after 3 rounds: the rest from round 1
        r1 = [q for q in json.load(open(f"{OUT}/round1.json"))["questions"] if BENCH.get(q) == s and q not in pick]
        pick += r1[:PER_SET - len(pick)]
    drawn[s] = pick
qs = [q for s in SETS5 for q in drawn[s]]
json.dump({"split": f"round{R}", "questions": qs, "made": time.strftime("%Y-%m-%d"), "source": "stop_judge.ipynb 7a"},
          open(f"{OUT}/round{R}.json", "w"))
json.dump({"questions": [{"id": q} for q in qs]}, open(f"{DAGDIR}/round{R}_qids.json", "w"))   # the harness's filter
assert not set(qs) & (test_ids | {q for q in QUESTION if norm(QUESTION[q]["question"]) in test_txt})
print(f"round {R}: {len(qs)} questions", {s: len(v) for s, v in drawn.items()},
      f"| already in another split or round: {len(set(qs) & taken)} (MetaQA reuse only)")

In [ ]:
# 7b. Round R's states, built exactly as the holdout's (4.4b): whole-question states of the round's questions, the
# oracle's label from the gold, render.py, validation, texts with both labels dropped, duplicates removed, pages (4.4).
R = int(os.environ.get("SJ_ROUND", "1")); RUN_R = f"sj2-round{R}"
round_qs = set(json.load(open(f"{OUT}/round{R}.json"))["questions"])
ST = pd.DataFrame([json.loads(l) for f in sorted(glob.glob(f"{V}/runs/dagger/round{R}_harvest/*.jsonl")) for l in open(f) if l.strip()])
ST["qid"] = ST.qid.astype(str); ST["goal"] = ST.goal.fillna("").astype(str)
ST["query"] = ST["query"] if "query" in ST else None
ST["is_subquestion"] = [norm(g) != norm(QUESTION.get(q, {}).get("question", "")) for g, q in zip(ST.goal, ST.qid)]
ST["dropped_for"] = ["not a round question" if q not in round_qs else "sub-question state" if sq else
                     "no candidate" if len(ids if ids is not None else []) == 0 else None
                     for q, sq, ids in zip(ST.qid, ST.is_subquestion, ST.ids)]
print(f"logged states: {len(ST)} on {ST.qid.nunique()} questions; dropped:", ST.dropped_for.value_counts().to_dict())
keep = ST[ST.dropped_for.isna()].copy()
need = collections.defaultdict(set)
for r in keep.itertuples():
    ids = [str(i) for i in r.ids]; q = r.query if isinstance(r.query, str) else None
    for i in ids[:30] + ids_in(q) + (ids_to_name(r.state, ids) if not q else []):
        if re.fullmatch(r"[mg]\.[0-9a-z_]+", i) and i not in NAMES: need["freebase"].add(i)
        elif "KG:" in i and i not in NAMES: need["biocq"].add(i)
for part in run_parallel(name_batch, [(g, sorted(v)[k:k + 200]) for g, v in need.items() for k in range(0, len(v), 200)], "naming"):
    NAMES.update({k: v for k, v in part.items() if k != "__error__" and v})
keep["label"] = [oracle([str(i) for i in ids], cand_names([str(i) for i in ids], list(nm if nm is not None else [])), q)[0]
                 for ids, nm, q in zip(keep.ids, keep.names, keep.qid)]
pairs = [from_kora(r.goal, r.state, [str(i) for i in r.ids], len(r.ids), r.query, NAMES) for r in keep.itertuples()]
keep["question"] = [q for q, _ in pairs]; keep["evidence"] = [e for _, e in pairs]
keep = keep[[len(validate(q, e)) == 0 for q, e in pairs]]
both = keep.groupby(["question", "evidence"]).label.transform("nunique") > 1
RD = keep[~both].drop_duplicates(["qid", "question", "evidence"])
RD = pd.DataFrame({"idx": range(len(RD)), "question": RD.question.values, "evidence": RD.evidence.values, "label": RD.label.astype(int).values,
                   "meta": [{"qid": q, "bench": BENCH[q], "source": f"round{R}", "site": s, "ts": float(t), "cand": str(c or "")}
                            for q, s, t, c in zip(RD.qid, RD.site, RD.ts, RD.cand)]})
os.makedirs(f"{V}/round{R}", exist_ok=True)
RD.to_parquet(f"{V}/round{R}/train.parquet", index=False)
TRACES[f"round{R}"] = [f"{RESULTS}/{RUN_R}/qwen3-30b-a3b/{s}-train/kora_nomem/traces" for s in SETS]
PAGED[f"round{R}"] = with_pages(f"{V}/round{R}/train.parquet", logged_calls(TRACES[f"round{R}"]))
rd = pd.read_parquet(f"{V}/round{R}/train.parquet")
rb = rd.assign(bench=rd.meta.map(lambda m: m["bench"]))
t = rb.groupby("bench").agg(states=("label", "size"), questions=("meta", lambda m: len({x["qid"] for x in m})), yes=("label", "mean"))
t.loc["total"] = [len(rb), rb.meta.map(lambda m: m["qid"]).nunique(), rb.label.mean()]
print(f"texts with both labels dropped: {int(both.sum())}; pages missing: {int((rd.meta.map(lambda m: m['page_round']) == 'none').sum())}")
t.round(3)

In [ ]:
# 7c. Round R's judge: a fresh Ettin 400M on A + B + rounds 1..R (a text in several sources kept once, the earliest;
# texts with both labels dropped), the frozen recipe, weights saved under models/dagger/round{R}.
R = int(os.environ.get("SJ_ROUND", "1")); DAG = f"{V}/models/dagger"
cols = ["question", "evidence", "label", "meta"]
mix = pd.concat([AB_TRAIN[cols]] + [pd.read_parquet(f"{V}/round{k}/train.parquet")[cols] for k in range(1, R + 1)], ignore_index=True)
both = mix.groupby(["question", "evidence"]).label.transform("nunique") > 1
DATA_R = mix[~both].drop_duplicates(["question", "evidence"], keep="first").reset_index(drop=True)
print(f"A+B {len(AB_TRAIN):,} + rounds 1..{R} -> {len(DATA_R):,} rows ({DATA_R.label.mean():.3f} yes); both labels dropped: {int(both.sum())}")
INFO_R = train_and_save(f"round{R}", DATA_R, None, f"A + B + rounds 1..{R}", root=DAG)
pd.Series(INFO_R["validation"] | {"threshold": INFO_R["threshold"], "rows": INFO_R["rows"], "minutes": INFO_R["minutes"]}).round(3)

In [ ]:
# 7d. Parity: round R's judge server must give the notebook's validation scores (200 logged validation states, sent as
# KORA sends them).
import urllib.request
R = int(os.environ.get("SJ_ROUND", "1")); JD = os.environ.get("SJ_JUDGE_DIR", f"{V}/models/dagger/round{R}")   # 7.8: another judge
URL = open(os.environ.get("SJ_URL_FILE", f"{V}/runs/dagger/round{R}_server.url")).read().strip()
VALS = pd.read_parquet(f"{V}/validation.parquet"); s_nb = np.load(f"{JD}/validation_scores.npy")
logged = {(str(r["qid"]), round(r["ts"], 3), str(r.get("cand") or "")): r
          for f in glob.glob(f"{V}/runs/val_oracle2/harvest/*.jsonl") for r in map(json.loads, open(f))}
rows = [k for k in random.Random(0).sample(range(len(VALS)), 400) if VALS.meta[k]["site"] == "leaf"][:200]
reqs = [logged[(VALS.meta[k]["qid"], round(VALS.meta[k]["ts"], 3), VALS.meta[k]["cand"] or "")] for k in rows]
body = {"states": [r["state"] for r in reqs], "questions": [r["goal"] for r in reqs],
        "extras": [{"ids": [str(i) for i in r["ids"]], "n": len(r["ids"]), "query": r.get("query"), "chain": []} for r in reqs]}
p = np.array(json.load(urllib.request.urlopen(urllib.request.Request(URL + "/score", data=json.dumps(body).encode(),
                                                headers={"Content-Type": "application/json"}), timeout=300))["p"])
thr = json.load(open(f"{JD}/judge.json"))["threshold"]; want = s_nb[rows]
same = float(np.mean((p >= thr) == (want >= thr)))
print(f"server {URL}: max |server - notebook| {np.abs(p - want).max():.4f}; same decision {same:.1%}")
assert same >= 0.98, "the server does not reproduce the notebook"

In [ ]:
# 7e. The gate: round R's live validation run against the best round so far (paired 95% intervals over questions, the
# authors' Hit rule and F1); keep it when F1 or Hit improves significantly; stop after 2 rounds without a gain or at 5.
import sys
RESULTS = f"{DATA_DIR}/results/eval"
sys.path.insert(0, f"{DATA_DIR}/stopjudge/eval_overlay_dag")
from harness.metrics.reasoning import hit_authors, live_files   # live shard files only (sN.jsonl)
R = int(os.environ.get("SJ_ROUND", "1")); LOG = f"{V}/runs/dagger/rounds.json"
V_IDS = sorted(q["id"] for q in json.load(open(f"{RAW}/qids/V_tune.json"))["questions"])
def live(run):
    rows = {}
    for f in [g for c in sorted(glob.glob(f"{RESULTS}/{run}/qwen3-30b-a3b/*-val/kora_nomem")) for g in live_files(c)]:
        for l in open(f):
            if l.strip():
                r = json.loads(l); rows[str(r["qid"])] = r
    def m(r):
        if r is None or r.get("error"): return 0.0, 0.0
        g = [a for a in (r.get("answers") or []) if a is not None and str(a).strip()] or [str(i) for i in (r.get("answer_ids") or [])]
        return hit_authors(r.get("answer") or "", g), float((r.get("eval") or {}).get("f1", 0) or 0)
    return np.array([m(rows.get(q)) for q in V_IDS]), len(rows)
history = [json.loads(l) for l in open(LOG)] if os.path.exists(LOG) else []
best = next((h for h in reversed(history) if h["best"]), {"round": 0, "gate_run": "sj2-live-AB400"})
cur, n_cur = live(f"sj2-gate-r{R}"); ref, _ = live(best["gate_run"])
rng = np.random.default_rng(0); d = cur - ref; idx = rng.integers(0, len(d), (10000, len(d))); boot = d[idx].mean(1)
lo, hi = np.percentile(boot, 2.5, axis=0), np.percentile(boot, 97.5, axis=0)
gain = bool(lo[1] > 0 or lo[0] > 0)
patience = 0 if gain else (history[-1]["patience"] + 1 if history else 1)
info = json.load(open(f"{V}/models/dagger/round{R}/judge.json"))
entry = {"round": R, "gate_run": f"sj2-gate-r{R}", "rows": n_cur, "hit": float(cur[:, 0].mean()), "f1": float(cur[:, 1].mean()),
         "against_round": best["round"], "hit_change": [float(d[:, 0].mean()), float(lo[0]), float(hi[0])],
         "f1_change": [float(d[:, 1].mean()), float(lo[1]), float(hi[1])], "best": gain, "patience": patience,
         "stop": bool(patience >= 2 or R >= 5), "offline_validation": info["validation"], "train_rows": info["rows"]}
open(LOG, "a").write(json.dumps(entry) + "\n")
pd.DataFrame([entry]).drop(columns=["offline_validation"]).T

**Rounds result:** two rounds on KORA, then the gate stopped. Round 1: live Hit .776 / F1 .668, against round 0's .800 / .694: −.024 [−.045, −.003] Hit, −.026 [−.045, −.008] F1. Round 2 (A+B + rounds 1–2, 127,500 rows): .782 / .676, −.018 [−.039, +.003] and −.019 [−.037, −.000]. Offline both are as good or better (round 2: FN/FP .065 / .068, AUROC .976), but neither beats round 0 live, so round 0's A+B judge stays the final judge (§9). Round 2's data: 1,000 fresh questions, 7,452 states, 19.3% yes.

### 7.8 KQA Pro: the adaptation round (one round of §7 on a dataset the judge never saw)
The base judge never saw KQA Pro (`round0/AB_noKQA`, §6.4). One round of §7 adapts it, and the 200 KQA Pro validation
questions (in no judge's training) measure it before and after:

| step | what |
|---|---|
| before | the base judge live on the 200 KQA Pro validation questions (KORA memory off), against the gold stop on them |
| rollout | KORA driven by the base judge on KQA Pro's 1,000 training questions, every state logged (`R=kqa1`) |
| data | every whole-question state, labelled by the oracle rule, built as 7b: `kqa_round1/train.parquet` |
| train | a fresh Ettin 400M on the base judge's rows (A+B without KQA Pro) + the round's rows, the frozen recipe; threshold balanced on the 666 validation states (no KQA Pro question read for it): `models/kqa/round1` |
| after | the adapted judge live on the same 200 questions; beside it the A+B judge, whose A has KQA Pro data generated from the graph and the gold (no agent run) |

**KQA Pro names.** The judge server, and the cells below, name KQA Pro's ids from its graph. Before,
the judge saw `KQAPro:3401 | kqapro.located_at_street_address | ...` where training showed `Barnard College | ...`; the KQA
Pro numbers of §6.4 were made that way and are replaced by these.

In [ ]:
# 7f-1. KQA Pro round R's states (SJ_KQA_ROUND=R), built as 7b: whole-question states of the 1,000 KQA Pro training
# questions (KORA driven by round R-1's judge; round 1: the judge that never saw KQA Pro), names from the graph (KQA Pro's
# included), the oracle's label, render.py, cleaned. Every round reuses the same 1,000 questions (all KQA Pro has here).
R_K = int(os.environ.get("SJ_KQA_ROUND", "1")); KQ, RUN_K = f"kqa_round{R_K}", f"sj2-kqa-round{R_K}"
EP["kqapro"] = open(f"{DATA_DIR}/runtime/endpoints/kqapro.txt").read().split()[0]

def kqa_names(ids):
    """KQA Pro's names: its graph has the BioKG layout under kqapro/ (rdfs:label); an id with no label stays an id."""
    try:
        vals = " ".join(f"<http://skill-lab-research.org/kqapro/node/{i}>" for i in ids)
        rows = sparql(EP["kqapro"], f"SELECT ?s ?n WHERE {{ VALUES ?s {{ {vals} }} ?s <http://www.w3.org/2000/01/rdf-schema#label> ?n }}")
        return {b["s"]["value"].rsplit("/node/", 1)[-1]: b["n"]["value"] for b in rows}
    except Exception as e:
        return {"__error__": outcome_of(e)}

round_qs = {str(x["id"]) for x in Q["kqapro"]}
ST = pd.DataFrame([json.loads(l) for f in sorted(glob.glob(f"{V}/runs/dagger/{KQ}_harvest/*.jsonl")) for l in open(f) if l.strip()])
ST["qid"] = ST.qid.astype(str); ST["goal"] = ST.goal.fillna("").astype(str)
ST["query"] = ST["query"] if "query" in ST else None
ST["is_subquestion"] = [norm(g) != norm(QUESTION.get(q, {}).get("question", "")) for g, q in zip(ST.goal, ST.qid)]
ST["dropped_for"] = ["not a round question" if q not in round_qs else "sub-question state" if sq else
                     "no candidate" if len(ids if ids is not None else []) == 0 else None
                     for q, sq, ids in zip(ST.qid, ST.is_subquestion, ST.ids)]
print(f"logged states: {len(ST)} on {ST.qid.nunique()} questions; dropped:", ST.dropped_for.value_counts().to_dict())
keep = ST[ST.dropped_for.isna()].copy()
need = set()
for r in keep.itertuples():
    ids = [str(i) for i in r.ids]; q = r.query if isinstance(r.query, str) else None
    need |= {i for i in ids[:30] + ids_in(q) + (ids_to_name(r.state, ids) if not q else []) if i.startswith("KQAPro:") and i not in NAMES}
need = sorted(need)
for part in run_parallel(kqa_names, [need[k:k + 200] for k in range(0, len(need), 200)], "naming KQA Pro"):
    NAMES.update({k: v for k, v in part.items() if k != "__error__" and v})
print(f"KQA Pro ids to name: {len(need)}; named: {sum(1 for i in need if i in NAMES)}")
keep["label"] = [oracle([str(i) for i in ids], cand_names([str(i) for i in ids], list(nm if nm is not None else [])), q)[0]
                 for ids, nm, q in zip(keep.ids, keep.names, keep.qid)]
pairs = [from_kora(r.goal, r.state, [str(i) for i in r.ids], len(r.ids), r.query, NAMES) for r in keep.itertuples()]
keep["question"] = [q for q, _ in pairs]; keep["evidence"] = [e for _, e in pairs]
keep = keep[[len(validate(q, e)) == 0 for q, e in pairs]]
both = keep.groupby(["question", "evidence"]).label.transform("nunique") > 1
RD = keep[~both].drop_duplicates(["qid", "question", "evidence"])
RD = pd.DataFrame({"idx": range(len(RD)), "question": RD.question.values, "evidence": RD.evidence.values, "label": RD.label.astype(int).values,
                   "meta": [{"qid": q, "bench": BENCH[q], "source": KQ, "site": s, "ts": float(t), "cand": str(c or "")}
                            for q, s, t, c in zip(RD.qid, RD.site, RD.ts, RD.cand)]})
os.makedirs(f"{V}/{KQ}", exist_ok=True)
RD.to_parquet(f"{V}/{KQ}/train.parquet", index=False)
TRACES[KQ] = [f"{RESULTS}/{RUN_K}/qwen3-30b-a3b/kqapro-train/kora_nomem/traces"]
PAGED[KQ] = with_pages(f"{V}/{KQ}/train.parquet", logged_calls(TRACES[KQ]))
rd = pd.read_parquet(f"{V}/{KQ}/train.parquet")
print(f"KQA Pro round: {len(rd):,} states on {rd.meta.map(lambda m: m['qid']).nunique()} questions, {rd.label.mean():.3f} yes; "
      f"texts with both labels dropped: {int(both.sum())}; by site {rd.meta.map(lambda m: m['site']).value_counts().to_dict()}")
for e in rd.evidence.sample(2, random_state=0):
    show(e[:400]); print("--")

In [ ]:
# 7f-2. KQA Pro round R's judge: a fresh Ettin 400M on the base judge's rows (A+B without KQA Pro, 6.4) + KQA Pro rounds
# 1..R (a text in several sources kept once, the earliest; texts with both labels dropped), the frozen recipe, weights saved.
R_K = int(os.environ.get("SJ_KQA_ROUND", "1"))
cols = ["question", "evidence", "label", "meta"]
mix = pd.concat([AB_NOKQA[cols]] + [pd.read_parquet(f"{V}/kqa_round{k}/train.parquet")[cols] for k in range(1, R_K + 1)],
                ignore_index=True)
both = mix.groupby(["question", "evidence"]).label.transform("nunique") > 1
DATA_K = mix[~both].drop_duplicates(["question", "evidence"], keep="first").reset_index(drop=True)
print(f"A+B without KQA Pro {len(AB_NOKQA):,} + KQA Pro rounds 1..{R_K} -> {len(DATA_K):,} rows ({DATA_K.label.mean():.3f} yes); "
      f"both labels dropped: {int(both.sum())}")
INFO_K = train_and_save(f"round{R_K}", DATA_K, None, f"A+B without KQA Pro + KQA Pro rounds 1..{R_K} (7.8)", root=f"{V}/models/kqa")
pd.Series(INFO_K["validation"] | {"threshold": INFO_K["threshold"], "rows": INFO_K["rows"], "minutes": INFO_K["minutes"]}).round(3)

In [ ]:
# 7f-3. KQA Pro's 200 validation questions (no judge trained on them), KORA memory off: the base judge (never saw KQA
# Pro), every round's judge so far, the A+B judge (KQA Pro data generated from the graph), the gold stop (upper bound);
# then round R's gate as 7e: kept when its Hit or F1 interval against the best round so far excludes zero; stop after
# 2 rounds without a gain or at round 4 (runs/dagger/kqa_rounds.json).
# Authors' Hit, exact-match Hit, F1, tokens; paired 95% intervals over questions; a missing or failed row is a miss.
import sys
sys.path.insert(0, f"{DATA_DIR}/stopjudge/eval_overlay_dag")
from harness.metrics.reasoning import hit_authors, live_files   # live shard files only (sN.jsonl)
RESULTS = f"{DATA_DIR}/results/eval"
KV = sorted(str(x["id"]) for x in QV["kqapro"])
plain = lambda x: re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", str(x).lower())).strip()

def kqa_rows(run, arm):
    out = {}
    for f in live_files(f"{RESULTS}/{run}/qwen3-30b-a3b/kqapro-val/{arm}"):
        for l in open(f):
            if l.strip():
                r = json.loads(l); out[str(r["qid"])] = r
    return out

def scored(r):
    if r is None or r.get("error"):
        return [0.0, 0.0, 0.0, np.nan]
    g = [a for a in (r.get("answers") or []) if a is not None and str(a).strip()] or [str(i) for i in (r.get("answer_ids") or [])]
    items = [x.strip() for x in re.split(r";|\n|, ", str(r.get("answer") or "")) if x.strip()]
    return [hit_authors(str(r.get("answer") or ""), g), float(any(plain(x) in {plain(y) for y in g} for x in items)),
            float((r.get("eval") or {}).get("f1", 0) or 0), float((r.get("fw_metrics") or {}).get("total_tokens") or 0)]

R_K = int(os.environ.get("SJ_KQA_ROUND", "1"))
BASE_K, AB_K, GOLD_K = "base judge (never saw KQA Pro)", "A+B judge (generated KQA Pro data)", "gold stop (upper bound)"
ARMS_K = {BASE_K: ("sj2-kqa-val-noKQA", "kora_nomem")}
ARMS_K.update({f"round {k} judge": (f"sj2-kqa-val-r{k}", "kora_nomem") for k in range(1, R_K + 1)})
ARMS_K.update({AB_K: ("sj2-kqa-val-AB", "kora_nomem"), GOLD_K: ("sj2-kqa-oracle", "kora_oracle_nomem")})
ROWS_K = {k: kqa_rows(*v) for k, v in ARMS_K.items()}
M_K = {k: np.array([scored(ROWS_K[k].get(q)) for q in KV]) for k in ARMS_K}
T_K = pd.DataFrame({k: {"done": len(ROWS_K[k]), "failed": sum(1 for q in KV if (ROWS_K[k].get(q) or {"error": 1}).get("error")),
                        "Hit (authors)": m[:, 0].mean(), "Hit (exact)": m[:, 1].mean(), "F1": m[:, 2].mean(),
                        "tokens": np.nanmean(m[:, 3])} for k, m in M_K.items() if len(ROWS_K[k])}).T

def paired(a, b, cols_=(0, 1, 2)):
    d = M_K[a][:, list(cols_)] - M_K[b][:, list(cols_)]; i = np.random.default_rng(0).integers(0, len(d), (10000, len(d)))
    lo, hi = np.percentile(d[i].mean(1), [2.5, 97.5], axis=0)
    return d.mean(0), lo, hi

cur = f"round {R_K} judge"
for a, b in [(f"round {k} judge", BASE_K) for k in range(1, R_K + 1)] + [(cur, GOLD_K), (cur, AB_K)]:
    mean, lo, hi = paired(a, b)
    print(f"{a} - {b}: " + "  ".join(f"{n} {mean[j]:+.3f} [{lo[j]:+.3f}, {hi[j]:+.3f}]" for j, n in enumerate(("Hit", "exact Hit", "F1"))))
LOGK = f"{V}/runs/dagger/kqa_rounds.json"
hist = [json.loads(l) for l in open(LOGK)] if os.path.exists(LOGK) else []
if not any(h["round"] == R_K for h in hist) and len(ROWS_K[cur]) == len(KV):
    best = next((h for h in reversed(hist) if h["best"]), {"round": 0})
    ref = BASE_K if best["round"] == 0 else f"round {best['round']} judge"
    mean, lo, hi = paired(cur, ref, (0, 2))
    gain = bool(lo[0] > 0 or lo[1] > 0)
    patience = 0 if gain else (hist[-1]["patience"] + 1 if hist else 1)
    entry = {"round": R_K, "hit": float(M_K[cur][:, 0].mean()), "f1": float(M_K[cur][:, 2].mean()), "against_round": best["round"],
             "hit_change": [float(mean[0]), float(lo[0]), float(hi[0])], "f1_change": [float(mean[1]), float(lo[1]), float(hi[1])],
             "best": gain, "patience": patience, "stop": bool(patience >= 2 or R_K >= 4)}
    open(LOGK, "a").write(json.dumps(entry) + "\n")
    print("gate:", entry)
T_K.round(3)

**KQA Pro adaptation result:** the 200 KQA Pro validation questions, KORA memory off. Base judge (never saw KQA Pro) Hit .505 / F1 .461. Round 1 (10,453 states on 999 training questions, 8.8% yes): .595 / .534, +.090 [+.030, +.150] Hit and +.072 [+.015, +.130] F1 against the base; its five-set validation is unchanged (AUROC .973). Rounds 2 and 3: .590 / .533 and .570 / .515, neither better than round 1; the gate stopped after round 3. The A+B judge, whose A has KQA Pro data generated from the graph (no agent run): .505 / .452, no better than the base. Gold stop .780 / .666. One round of the pipeline adapts the judge to a dataset it never saw; more rounds on the same 1,000 questions add nothing, and .185 Hit to the gold stop remains.

## 8. Transfer: the judge inside other systems
The judge, trained on KORA's states and generated rows only (round 0's A+B, §6.2), replaces one step in the authors'
code of three systems: the call that decides whether to stop. Nothing else changes: their search, pruning, prompts,
decoding settings and fallbacks run as published. Each system runs three ways on the 666 validation questions (Qwen3-30B):

| arm | the stop | where |
|---|---|---|
| their judge | the system as published | `sj2-val-asis` (ToG, PoG, CoR) |
| our judge | the judge at the A+B threshold (.0658), zero-shot: never trained on that system's states | `sj2-val-estimator` (ToG, CoR), `sj2-val-fix-estimator` (PoG, after the path fix) |
| gold | the gold decides over the same candidates (the oracle rule): the best that stop decision could do | `sj2-val-estimator` |

The hooks (the harness's baselines): **ToG** and **PoG**: their per-depth evaluation
call is replaced; each kept (topic, relation, direction) group of the depth is a candidate, with all the entities their
search returned for it and the hops that led to the topic as its path, written in KORA's query dialect so the judge reads
it as in training; accepted, the system stops and answers with that group's names. **CoR**: at each decision the
executed chain's whole result set is the candidate; accepted, CoR stops with it, otherwise its own stop becomes a forward
step. **KORA**: its own LLM judge (current version, memory off) against the same judge, every other setting equal.
**Search-on-Graph** has no separate stop to replace; it is shown as published.

Metrics: Hit by the ToG / PoG / SoG authors' rule (an answer and a gold contain each other, `harness/metrics/reasoning.hit_authors`),
exact-match Hit (an answer item equals a gold after normalisation), F1 (`eval.f1`), tokens per question; a failed
question (mostly the 200 s timeout) is a miss. One run each; repeats in 8d.

In [ ]:
# 8. Helpers: every arm's rows (the harness's live shard files, one row per question), the metrics, paired bootstraps.
import sys
sys.path.insert(0, f"{DATA_DIR}/stopjudge/eval_overlay_dag")
from harness.metrics.reasoning import hit_authors, live_files   # live shard files only (sN.jsonl)
RESULTS = f"{DATA_DIR}/results/eval"
V_TUNE = {q["id"] for q in json.load(open(f"{RAW}/qids/V_tune.json"))["questions"]}
SETS8, NO_METAQA = ["webqsp", "cwq", "grailqa", "metaqa", "biocq"], ["webqsp", "cwq", "grailqa", "biocq"]
plain = lambda x: re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", str(x).lower())).strip()

def arm_rows(run, arm, sets=SETS8):
    out = {}
    for s in sets:
        for f in live_files(f"{RESULTS}/{run}/qwen3-30b-a3b/{s}-val/{arm}"):
            for l in open(f):
                if l.strip():
                    r = json.loads(l)
                    if str(r["qid"]) in V_TUNE:
                        out[(s, str(r["qid"]))] = r
    return out

UNIVERSE = {s: sorted({q for (s2, q) in arm_rows("sj2-val-asis", "tog_official", [s])}) for s in SETS8}
golds_of = lambda r: [a for a in (r.get("answers") or []) if a is not None and str(a).strip()] or [str(i) for i in (r.get("answer_ids") or [])]
items_of = lambda a: [x.strip() for x in re.split(r";|\n|, ", str(a or "")) if x.strip()]

def scores(r, whole_reply=False):
    """(authors' Hit, exact-match Hit, F1, tokens) of a row; a missing or failed row is a miss. whole_reply: a ToG answer
    that is exactly "Yes" is scored on its whole reply instead (the format failure of 8c)."""
    if r is None or r.get("error"):
        return [0.0, 0.0, 0.0, np.nan]
    a = str(r.get("answer") or "")
    if whole_reply and a.strip().lower() == "yes":
        a = re.sub(r"\{\s*yes\s*\}", " ", (r.get("fw_metrics") or {}).get("tog_reply", ""), flags=re.I)
    g = golds_of(r)
    return [hit_authors(a, g), float(any(plain(x) in {plain(y) for y in g} for x in items_of(r.get("answer")))),
            float((r.get("eval") or {}).get("f1", 0) or 0), float((r.get("fw_metrics") or {}).get("total_tokens") or 0)]

def paired(d, n=10000):
    d = np.asarray(d, float); i = np.random.default_rng(0).integers(0, len(d), (n, len(d))); b = d[i].mean(1)
    return d.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)

ARMS8 = [("ToG", "their judge", "sj2-val-asis", "tog_official", SETS8), ("ToG", "our judge", "sj2-val-estimator", "tog_estimator", SETS8),
         ("ToG", "gold", "sj2-val-estimator", "tog_estimator_oracle", SETS8),
         ("PoG", "their judge", "sj2-val-asis", "pog_official", SETS8), ("PoG", "our judge", "sj2-val-fix-estimator", "pog_estimator", SETS8),
         ("PoG", "gold", "sj2-val-estimator", "pog_estimator_oracle", SETS8),
         ("CoR", "their judge", "sj2-val-asis", "cor", NO_METAQA), ("CoR", "our judge", "sj2-val-estimator", "cor_estimator", NO_METAQA),
         ("CoR", "gold", "sj2-val-estimator", "cor_estimator_oracle", NO_METAQA),
         ("KORA", "their judge", "sj2-val-kora-llm", "kora_nomem", SETS8), ("KORA", "our judge", "sj2-live-AB400", "kora_nomem", SETS8),
         ("KORA", "gold", "sj2-val-oracle2", "kora_oracle_nomem", SETS8),
         ("SoG", "their stop", "sj-val-sog", "sog_official", SETS8)]
ROWS8 = {(sy, who): (arm_rows(run, arm, sets), sets) for sy, who, run, arm, sets in ARMS8}
keys_of = lambda sets: [(s, q) for s in sets for q in UNIVERSE[s]]
M8 = {k: np.array([scores(rows.get(q)) for q in keys_of(sets)]) for k, (rows, sets) in ROWS8.items()}
print({f"{k[0]} {k[1]}": f"{len(ROWS8[k][0])}/{len(keys_of(ROWS8[k][1]))}" for k in ROWS8})

In [ ]:
# 8a. The table: every system's three stops; per-set columns are authors' Hit.
T8 = []
for (sy, who), (rows, sets) in ROWS8.items():
    m = M8[(sy, who)]; ks = keys_of(sets)
    row = {"system": sy, "stop": who, "Hit": m[:, 0].mean(), "exact Hit": m[:, 1].mean(), "F1": m[:, 2].mean(),
           "tokens (k)": np.nanmean(m[:, 3]) / 1000, "failed": sum(1 for q in ks if rows.get(q) is None or rows[q].get("error"))}
    row.update({s: np.mean([scores(rows.get((s, q)))[0] for q in UNIVERSE[s]]) if s in sets else np.nan for s in SETS8})
    T8.append(row)
T8 = pd.DataFrame(T8).set_index(["system", "stop"])
T8.round(3)

In [ ]:
# 8b. Our judge against each system's own judge and against its gold, paired over questions (95% intervals); the
# relative change is against their judge's score.
G8 = []
for sy in ("ToG", "PoG", "CoR", "KORA"):
    own, ours, gold = M8[(sy, "their judge")], M8[(sy, "our judge")], M8[(sy, "gold")]
    for j, name in ((0, "Hit"), (1, "exact Hit"), (2, "F1")):
        mean, lo, hi = paired(ours[:, j] - own[:, j]); gm, glo, ghi = paired(ours[:, j] - gold[:, j])
        G8.append({"system": sy, "metric": name, "their judge": own[:, j].mean(), "our judge": ours[:, j].mean(),
                   "gold": gold[:, j].mean(), "ours - theirs": f"{mean:+.3f} [{lo:+.3f}, {hi:+.3f}]",
                   "relative": f"{mean / own[:, j].mean():+.1%}", "ours - gold": f"{gm:+.3f} [{glo:+.3f}, {ghi:+.3f}]"})
pd.DataFrame(G8).set_index(["system", "metric"]).round(3)

In [ ]:
# 8c. How each arm's answer is read, checked: answers that are a bare verdict or empty, and misses whose raw reply holds
# a gold (an upper bound on what the extraction loses). ToG's own judge: replies "{Yes}. ... the answer is X" without a
# second {...}, so the authors' extraction returns "Yes"; scored on the whole reply, in ToG's favour, below.
RAW_FIELD = {"tog_official": "tog_reply", "tog_estimator": "tog_reply", "pog_official": "pog_reply", "pog_estimator": "pog_reply",
             "sog_official": "sog_reply"}
sq = lambda s: re.sub(r"\s+", "", str(s).lower())
A8 = []
for sy, who, run, arm, sets in ARMS8:
    rows, _ = ROWS8[(sy, who)]; ok = [r for r in rows.values() if not r.get("error")]
    bare = [r for r in ok if re.sub(r"\W+", "", str(r.get("answer") or "").lower()) in ("", "yes", "no", "null", "none")]
    miss = [r for r in ok if not hit_authors(str(r.get("answer") or ""), golds_of(r))]
    f = RAW_FIELD.get(arm)
    lost = sum(1 for r in miss if f and any(sq(g) and sq(g) in sq((r.get("fw_metrics") or {}).get(f, "")) for g in golds_of(r)))
    A8.append({"system": sy, "stop": who, "answered": len(ok), "bare verdict or empty": len(bare),
               "of which Yes": sum(1 for r in bare if str(r.get("answer")).strip().lower() == "yes"),
               "misses": len(miss), "misses with a gold in the reply": lost if f else None,
               "answer items, median": np.median([len(items_of(r.get("answer"))) for r in ok])})
display(pd.DataFrame(A8).set_index(["system", "stop"]))
rows, sets = ROWS8[("ToG", "their judge")]
whole = np.array([scores(rows.get(q), whole_reply=True) for q in keys_of(sets)])
mean, lo, hi = paired(M8[("ToG", "our judge")][:, 0] - whole[:, 0])
print(f"ToG's own judge with each 'Yes' read from its whole reply: Hit {whole[:, 0].mean():.3f} (authors' extraction "
      f"{M8[('ToG', 'their judge')][:, 0].mean():.3f}); per set "
      + ", ".join(f"{s} {np.mean([scores(rows.get((s, q)), True)[0] for q in UNIVERSE[s]]):.3f}" for s in SETS8)
      + f"; our judge - that: {mean:+.3f} [{lo:+.3f}, {hi:+.3f}]")

In [ ]:
# 8d. Repeats: the same arm run again (the model servers' load differs between runs), paired over questions.
REP8 = [("KORA, our judge", ("sj2-live-AB400", "kora_nomem", SETS8), ("sj2-live-AB400-r2", "kora_nomem", SETS8)),
        ("ToG, our judge", ("sj2-val-estimator", "tog_estimator", SETS8), ("sj2-val-rep-estimator", "tog_estimator", SETS8)),
        ("PoG, our judge (before the path fix)", ("sj2-val-estimator", "pog_estimator", SETS8), ("sj2-final-estimator", "pog_estimator", SETS8)),
        ("CoR, our judge", ("sj2-val-estimator", "cor_estimator", NO_METAQA), ("sj2-final-estimator", "cor_estimator", NO_METAQA))]
R8 = []
for name, a, b in REP8:
    ra, rb = arm_rows(*a), arm_rows(*b); ks = keys_of(a[2])
    ma, mb = np.array([scores(ra.get(q)) for q in ks]), np.array([scores(rb.get(q)) for q in ks])
    row = {"arm": name, "questions": len(ks)}
    for j, m in ((1, "exact Hit"), (2, "F1")):
        mean, lo, hi = paired(mb[:, j] - ma[:, j])
        row.update({f"{m} run 1": ma[:, j].mean(), f"{m} repeat": mb[:, j].mean(), f"{m} repeat - run 1": f"{mean:+.3f} [{lo:+.3f}, {hi:+.3f}]"})
    R8.append(row)
pd.DataFrame(R8).set_index("arm").round(3)

**Transfer result:** zero-shot, the judge beats each system's own judge on F1 and exact-match Hit.

- **F1:** PoG +.080 [+.053, +.108], CoR +.097 [+.060, +.134], KORA +.160 [+.130, +.190]. ToG +.180 [+.146, +.213], but its own judge's F1 is not comparable (8c).
- **Hit:** CoR +.093 [+.051, +.134], KORA +.108 [+.075, +.141]. PoG's Hit is level (+.014 [−.020, +.047]), as is its gold stop's (.643). ToG: between +.036 [+.003, +.071], with its own judge's "Yes" replies read in full in its favour, and +.180, with the authors' extraction.
- **Per set, the own judge stays ahead** on WebQSP for PoG and CoR, and on MetaQA for PoG. There their LLM writes right answers that are not among the retrieved candidates: PoG's own judge scores .776 on WebQSP, above its gold stop's .744.
- **KORA with the judge:** .800 / .694 at 16.0k tokens a question, against Search-on-Graph's .745 / .639 at 65.4k. Against KORA's own gold stop: Hit −.039 [−.065, −.014], F1 level (−.014 [−.037, +.008]).
- **Repeats** move by .015 or less (8d).

**Caveats**:
1. ToG's contrast is not only the stop: once a group is accepted, the hook answers with that group's names, while ToG's own judge answers in prose.
2. The threshold was balanced on the states of these same 666 questions, so these are validation numbers, not held-out ones. The test reads come with the experimental design.
3. KORA's gold arm runs without the clock fallback that the other two KORA arms use.
4. Hit credits any one item, and CoR's accepted sets are longer (11.5 items on average against 3.1), so F1 and exact-match Hit carry the claim.
5. Tokens are means over answered questions, so arms with more timeouts look cheaper.

**Reading rule:** only the live shard files count (`sN.jsonl`), not the retried attempts in `sN.failed.jsonl`.

## 9. The final judge
| | |
|---|---|
| model | `models/round0/AB`: Ettin 400M (encoder), `cls` pooling, two classes, fine-tuned on A + B (112,589 rows) with the recipe of §6.1d (lr 3e-5, 2 epochs, batch 32, AdamW with decoupled weight decay, 6% warm-up, linear decay, bf16) |
| input | the (question, evidence) pair of `render.py`: the path as subject / relation / object lines and the candidate answer (the schema page is an ablation, §6.1) |
| threshold | .0658, balancing FN and FP on the 666 validation questions' states |
| offline | validation FN/FP .073 / .073, AUROC .976; holdout (read once) FN/FP .062 / .081, AUROC .979 (§6.3) |
| live | KORA memory off: Hit .800 / F1 .694 (gold stop .839 / .709); inside ToG, PoG and CoR zero-shot, §8 |
| a new dataset | one round of §7 on its training questions adapts it (KQA Pro: +.090 Hit, §7.8; `models/kqa/round1`) |

**The released estimator** is this recipe trained on A + B plus copies of a random third of the rows with every entity
name replaced by an invented one (`python -m estimator.train.rename`), so names carry no signal; threshold .0494.

**Serving.** `python -m estimator.server --checkpoint DIR` loads the checkpoint on a GPU (bf16) and answers
`POST /score {states, questions, extras}` with P(yes) per state. `extras` carries what the renderer needs: the candidate ids
(first 30), their count, the executed query and the chain of queries the task stands on. The server names every id from
the graph (Freebase in this release), renders the pair with `render.py` exactly as the training rows were made, and
scores it; a state it cannot render scores 0.0 (keep searching). KORA calls it from `kora/modules/sufficiency/estimator.py`
(`KORA_STOP=estimator`, `ESTIMATOR_URL`, `ESTIMATOR_TAU`) at the leaf and at the root; the other systems' hooks
post the same body. A parity check (7d) runs before every live run: the server must reproduce the notebook's validation
decisions.